# KV260 icin YOLOX-Tiny - Gemi Tespiti (Kaggle)

**Hedef:** tek sinif `ship`. Kamera artik havadan degil, gemiyle yaklasik
ayni mesafede/yukseklikte (kiyi, iskele, gemi guvertesi). Girdi termal veya
gri ton olabilir; egitim buna gore alan-saglamligi augmentasyonu icerir
(bkz. `yolox_tiny_ship.py` basligi).

**Bagli olmasi gereken Kaggle veri seti** (Add Input): 6 ham kaynak zip'i
(`tools/prepare_ship_kaggle_upload.py` ile hazirlanir) -- hangi klasor/dataset
adiyla yuklendigi ONEMLI DEGIL, asagidaki kesif hucresi kaynaklari klasor
adindan degil **kategori imzasindan** tanir.

Ayarlar: **Accelerator = GPU**, **Internet = On**.


In [ ]:
import os
from pathlib import Path

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
%cd {WORK}
!nvidia-smi

EXP_FILE = "yolox_tiny_ship.py"
INIT_URL = ("https://github.com/Megvii-BaseDetection/YOLOX/releases/download/"
            "0.1.1rc0/yolox_tiny.pth")
print("exp     :", EXP_FILE)
print("baslangic agirligi:", INIT_URL.rsplit("/", 1)[-1])


In [ ]:
# YOLOX kurulumu (Kaggle'daki hazir torch surumune dokunmadan).
# Kaggle ve Vitis AI VM ayni commit'i kullanir; main dali kullanilmaz.
import importlib
import site
import subprocess
import sys

YOLOX_COMMIT = "6ddff4824372906469a7fae2dc3206c7aa4bbaee"
YOLOX_DIR = Path(WORK) / "YOLOX"

if not YOLOX_DIR.is_dir():
    !git clone --filter=blob:none --no-checkout https://github.com/Megvii-BaseDetection/YOLOX.git "{YOLOX_DIR}"
!git -C "{YOLOX_DIR}" fetch --depth 1 origin {YOLOX_COMMIT}
!git -C "{YOLOX_DIR}" checkout --detach {YOLOX_COMMIT}
current = !git -C "{YOLOX_DIR}" rev-parse HEAD
assert current and current[0] == YOLOX_COMMIT, f"Yanlis YOLOX commit'i: {current}"


def _pip(*args):
    """pip'i cagirir. `!pip` kabuk cagrisinin aksine hatayi yutmaz."""
    proc = subprocess.run([sys.executable, "-m", "pip", *args],
                          capture_output=True, text=True)
    if proc.returncode != 0:
        print(proc.stdout[-4000:])
        print(proc.stderr[-4000:])
    return proc.returncode


# --no-build-isolation sart: YOLOX'un setup.py'si torch'u import eder, pip'in
# izole build ortaminda torch bulunmaz ve kurulum sessizce basarisiz olur.
rc = _pip("install", "--no-deps", "--no-build-isolation", "-e", str(YOLOX_DIR))
if rc != 0:
    print("Editable kurulum basarisiz; editable olmayan kuruluma dusuluyor.")
    rc = _pip("install", "--no-deps", "--no-build-isolation", str(YOLOX_DIR))
assert rc == 0, "YOLOX kurulumu basarisiz (yukaridaki pip ciktisina bakin)."

assert _pip("install", "-q", "loguru", "tabulate", "psutil", "pycocotools",
            "thop", "ninja", "albumentations") == 0, "Yardimci paket kurulumu basarisiz."

# pip'in yazdigi .pth dosyalari yalnizca yorumlayici acilisinda okunur; calisan
# kernel'in sys.path'ini elle tazelemezsek import ayni oturumda basarisiz olur.
for _site_dir in getattr(site, "getsitepackages", list)():
    site.addsitedir(_site_dir)
if str(YOLOX_DIR) not in sys.path:
    sys.path.insert(0, str(YOLOX_DIR))
importlib.invalidate_caches()

# numpy 1.24+ uyumlulugu: kaldirilan eski takma adlar icin shim
import numpy as np
for _alias, _type in (("float", float), ("int", int), ("bool", bool)):
    if _alias not in np.__dict__:
        setattr(np, _alias, _type)

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
import yolox
print("yolox:", yolox.__version__, "|", yolox.__file__)
import albumentations
print("albumentations:", albumentations.__version__)


In [ ]:
%%writefile dataset_common.py
#!/usr/bin/env python3
"""Veri seti birlestiricilerin paylastigi format-bagimsiz yardimcilar.

Buraya YALNIZCA hicbir kaynagin (VisDrone, Roboflow, VOC, ...) semantigini
bilmeyen kod girer: arsiv okuma, kutu kirpma, oturum bazli bolme, goruntu
cikarma. Kaynaga ozel esleme ve okuyucular cagiran modulde kalir.

Neden ayri modul: gemi hatti (tools/build_ship_dataset.py) bu yardimcilarin
9'unu kullaniyordu ve bunun icin havadan/VisDrone birlestiricisinin tamamini
import etmek zorundaydi -- Kaggle not defterine de o dosyanin tamami
gomuluyordu. Gemi isi drone verisi kullanmiyor; bagimlilik da kullanmamali.

Kullanici: tools/build_ship_dataset.py. (Havadan/VisDrone birlestiricisi bu
projeden cikarildi; oturum bazli bolmenin grup-sayisina gore calisan eski
surumleri -- resplit_by_group, carve_test_split -- onunla birlikte silindi.
Gerekirse git gecmisindeki tools/build_dataset.py icinde bulunur.)
"""

import io
import random
import re
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

#: Bolme rastgeleligi tekrarlanabilir olsun diye sabit.
SPLIT_SEED = 1337


# --------------------------------------------------------------------------
# zip veya klasor farkini gizleyen ince katman
# --------------------------------------------------------------------------
class Archive:
    """Bir .zip dosyasini veya bir klasoru ayni arayuzle okur."""

    def __init__(self, path):
        self.path = Path(path)
        self._zip = None
        if self.path.suffix.lower() == ".zip":
            self._zip = zipfile.ZipFile(self.path)
            self._names = [n for n in self._zip.namelist() if not n.endswith("/")]
        elif self.path.is_dir():
            self._names = [
                p.relative_to(self.path).as_posix()
                for p in self.path.rglob("*") if p.is_file()
            ]
        else:
            raise SystemExit(f"HATA: bulunamadi veya desteklenmiyor: {path}")
        self._name_set = set(self._names)

    def names(self):
        return self._names

    def read(self, name):
        if self._zip is not None:
            return self._zip.read(name)
        return (self.path / name).read_bytes()

    def open(self, name):
        if self._zip is not None:
            return self._zip.open(name)
        return open(self.path / name, "rb")

    def exists(self, name):
        return name in self._name_set

    def image_size(self, name):
        """Yalnizca basligi okur; tum goruntuyu cozmez."""
        # Bu modulun bolme/manifest yardimcilari Pillow gerektirmez. Importu
        # burada tutmak, yalnizca mevcut COCO JSON'larini denetleyen veya
        # yeniden bolen araclarin gereksiz goruntu kutuphanesi bagimliligiyla
        # acilista durmasini engeller.
        from PIL import Image
        with self.open(name) as handle:
            with Image.open(io.BytesIO(handle.read()) if self._zip else handle) as im:
                return im.size  # (width, height)

    def close(self):
        if self._zip is not None:
            self._zip.close()
            self._zip = None

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        self.close()
        return False


class Record:
    """Birlestirilmis veri setindeki tek bir goruntu."""

    __slots__ = ("source", "member", "file_name", "width", "height",
                 "split", "anns", "ignore_regions", "group", "origin")

    def __init__(self, source, member, file_name, width, height, split, group):
        self.source = source          # kaynak adi (visdrone, vesselimg, ...)
        self.member = member          # arsiv icindeki yol
        # Goruntunun geldigi arsiv. Kaynak adina gore eslemek yetmez:
        # visdrone train ve val ayni kaynak adini paylasir ama farkli
        # zip'lerde durur.
        self.origin = None
        self.file_name = file_name    # cikti COCO'daki goreli yol
        self.width = width
        self.height = height
        self.split = split            # "train" | "val"
        self.group = group            # sizinti kontrolu icin oturum anahtari
        self.anns = []                # {bbox, category_id, iscrowd, ignore}
        self.ignore_regions = []


def clip_box(x, y, w, h, width, height):
    """Kutuyu goruntuye kirpar; gecersizse None doner."""
    x1 = max(0.0, float(x))
    y1 = max(0.0, float(y))
    x2 = min(float(width), float(x) + float(w))
    y2 = min(float(height), float(y) + float(h))
    if x2 - x1 <= 0 or y2 - y1 <= 0:
        return None
    return [x1, y1, x2 - x1, y2 - y1]


def has_part(name, part):
    """Yol bileseni tam eslesme ile aranir.

    Arsiv kokunun nereye isaret ettigine gore ayni dosya
    'VisDrone2019-DET-train/images/x.jpg' veya 'images/x.jpg' olarak
    gorunebilir; '/images/' arayan bir kontrol ikincisini kaciririrdi.
    """
    return part in name.split("/")


def swap_part(name, old, new):
    parts = name.split("/")
    return "/".join(new if p == old else p for p in parts)


def session_key(file_name, source=""):
    """Ayni cekimden / ayni kaynak goruntuden gelen kareleri gruplayan anahtar.

    Roboflow dosya adlari '<taban>_<kare>_jpg.rf.<hash>.jpg' bicimindedir;
    taban kisim ya cekim oturumunu (kamera + zaman damgasi) ya da augment
    kopyalarinin turedigi kaynak goruntuyu tasir. Ikisi de ayni bolumde
    kalmalidir.

    Anahtar kaynak adiyla oneklenir: iki veri seti de goruntulerini 1, 2, 3
    diye numaraladigi icin onek olmadan alakasiz goruntuler ayni oturum
    sayilirdi. (Kontrol edildi: milrec ve mendeley'deki ayni adli goruntuler
    gorsel olarak tamamen farkli.)
    """
    base = file_name.rsplit("/", 1)[-1]
    prefix = f"{source}:" if source else ""
    match = re.match(r"(.+?)_\d+_(?:jpg|png)\.rf\.", base)
    if match:
        return prefix + match.group(1)
    match = re.match(r"(.+?)\.rf\.", base)
    if match:
        return prefix + match.group(1)
    return prefix + base


def split_by_group_targets(records, val_fraction, test_fraction, seed=SPLIT_SEED):
    """Gruplari BOLMEDEN, **goruntu sayisi** hedefine gore train/val/test ayirir.

    resplit_by_group()'tan farki hedefin ne oldugu: o, grup SAYISININ bir
    oranini val'e atar. Gruplar cok esitsizse gerceklesen goruntu orani
    hedeften sapar -- olculdu (2026-08-12): hedef %25 val iken
    vais_smd_marvel'de %10,6, sea_vessels'ta %12,0 cikmisti. Burada kota
    goruntu cinsinden tutulur, o yuzden gerceklesen oran hedefe yakin kalir.

    Hedefi asacak grup ATLANIR, bolunmez: bir oturumun (video, kamera akisi,
    ayni fotografin augment kopyalari) yarisini train'e yarisini val'e koymak
    tam da onlemeye calistigimiz sizintidir. Sonucta cok buyuk bir grup
    (or. tek bir IP kamera akisi) her zaman train'de kalir; val o kamerayi
    hic gormez -- istenen davranis budur, val'in bagimsizligi oran
    hassasiyetinden onemli.

    Donus: {"train": n, "val": n, "test": n} goruntu sayilari.
    """
    groups = defaultdict(list)
    for record in records:
        groups[record.group].append(record)
    names = sorted(groups)
    rng = random.Random(seed)
    rng.shuffle(names)

    total = len(records)
    assigned = {}
    for split, fraction in (("val", val_fraction), ("test", test_fraction)):
        target = max(1, round(total * fraction))
        taken = 0
        for name in names:
            if name in assigned or taken >= target:
                continue
            size = len(groups[name])
            if taken + size > target * 1.3:
                continue
            assigned[name] = split
            taken += size
        if taken == 0:
            # Hicbir grup kotaya sigmadi (tek dev grup gibi): bolme yapmamak
            # yerine en kucugunu al -- bos val/test sessiz bir hata olurdu.
            free = [n for n in names if n not in assigned]
            if free:
                assigned[min(free, key=lambda n: len(groups[n]))] = split

    counts = Counter()
    for record in records:
        record.split = assigned.get(record.group, "train")
        counts[record.split] += 1
    return dict(counts)


def extract_images(records, images_out):
    """Goruntuleri COCO `file_name` alanlariyla ortusen duzene cikarir.

    Sonuc: <images_out>/<kaynak>/<ad>.jpg  ->  YOLOX tarafinda
    data_dir=<ust dizin>, name="images" ile dogrudan okunur.
    Var olan dosyalar atlanir; yarim kalan cikarma guvenle tekrarlanabilir.
    """
    # Ayni kayit tekrar katsayisi yuzunden birden fazla kez gelebilir.
    unique = {r.file_name: r for r in records}
    by_origin = defaultdict(list)
    for rec in unique.values():
        by_origin[str(rec.origin)].append(rec)

    written = skipped = 0
    for origin in sorted(by_origin):
        with Archive(origin) as archive:
            for rec in sorted(by_origin[origin], key=lambda r: r.file_name):
                target = images_out / rec.file_name
                if target.exists() and target.stat().st_size > 0:
                    skipped += 1
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(rec.member))
                written += 1
        print(f"   {Path(origin).name}: cikarildi")
    print(f"goruntuler: {written} yazildi, {skipped} zaten vardi -> {images_out}")


In [ ]:
%%writefile build_ship_dataset.py
#!/usr/bin/env python3
"""6 kaynagi tek sinifli ("ship") bir COCO veri setinde birlestirir.

Bu, havadan land+sea 2 sinifli projeden AYRI bir pivot: kamera artik gemiyle
yaklasik ayni mesafede/yukseklikte (kiyi, iskele, gemi guvertesi), yukaridan
degil. Termal/gri ton goruntu de kabul edilir -- kullanicinin acik istegi.

Hedef sinif
-----------
  1 = ship (tum gemi/tekne tipleri tek sinifta)

Kaynaklar (hepsi Roboflow COCO export'u, WUTDet haric)
-------------------------------------------------------
  vais_smd_marvel     VAIS+SMD+MARITIME+WSODD+MARVEL birlesimi (Roboflow)
  singapore_maritime  Singapore Maritime Dataset yeniden-export (Roboflow)
  sea_vessels         Sea Vessels Dataset v2 (Roboflow, kucuk, augment'li)
  ship_model          "ship model" v4 (Roboflow, buyuk, tek sinif)
  ir_thermal          ir.v1i (Roboflow, GERCEK termal, 7 gemi tipi)
  wutdet              WUTDet Part A (Pascal VOC XML, 100K'lik setin bir parcasi)

Kategori eslemesi neden tablo, neden anahtar-kelime degil
-----------------------------------------------------------
Her kaynagin kategorileri elle acilip incelendi ("gemi gibi gorunen" ada
gore degil, gercek kutu kullanimina bakilarak -- bkz. VAIS'teki "object"
adli gercek gemi sinifi, veya kullanilmayan "root" kategoriler). Sonuc
CLASS_MAPS tablosuna yazildi. Okuyucu, tabloda olmayan bir kategoriyle
karsilasirsa **sessizce atlamaz, hata verir** -- yeni bir Roboflow surumu
kategori eklerse fark edilsin diye.

Kaynaklar arasi cakisma (kritik, olculdu)
-------------------------------------------
2026-08-12'de tum arsivler uzerinde yeniden olculdu (12.057 goruntu atiliyor):
  * ship_model  -> vais_smd_marvel   9271  ayni MARVEL fotografi (biri gri
                                           tona cevrilmis). 200 rastgele
                                           ciftin 200'u pikselde ayni cikti.
  * singapore   -> vais_smd_marvel    956  ayni SMD video karesi (40/40 ayni)
  * ship_model  -> ship_model         1830 KAYNAK ICI kopya: ayni fotograf
                                           zip'te iki cozunurlukte (orijinal
                                           + Roboflow'un 640x640 kopyasi);
                                           60/60 ayni cikti.
Bu YALNIZCA bir tekrar sorunu degil: ele alinmazsa ayni fotografin biri
train'e biri val/test'e dusup sizinti yaratabilir. cross_source_identity()
bu kimligi yakalar, dedupe_sources() zengin kaynaklari once isleyip
VAIS-merge'den zaten alinmis olani cikarir.

Bilinen sinir: kimlik, dosya adinin bastaki sayisindan uretiliyor. MARVEL
adlari 6 haneli sifir dolgulu ("000027") oldugu icin guvenli, ama dolgusuz
kisa sayilarda ("10_jpg") cakisabiliyor. Olculdu: 909 kisa kimlikten
orneklemede ~%2 yanlis eslesme, yani ~15 goruntu (verinin %0,02'si) bosuna
atiliyor. Hata yonu guvenli tarafta (fazla atar, sizdirmaz).

Bolme politikasi -- neden kaynaklarin kendi train/valid/test'i kullanilmiyor
-----------------------------------------------------------------------------
Roboflow export'lari kareleri RASTGELE bolmus: ayni video/oturum ucunde de
bulunuyor. Bu haliyle dogrulama skoru sahte cikar, o yuzden bolme burada
sifirdan yapilir: her kaynak AYRI, oturum bazli, hedef 70/15/15 (goruntu
sayisi). Oturum anahtari icin bkz. ship_session_key() -- 2026-08-12'de
olculen sizinti oradaki kalip eksikliginden kaynaklaniyordu.

Lisans
------
  vais_smd_marvel      CC BY 4.0
  singapore_maritime   Public Domain
  sea_vessels          CC BY 4.0
  ship_model           CC BY 4.0
  ir_thermal           CC BY 4.0
  wutdet               Belirsiz -- indirilen Part A arsivinde lisans dosyasi
                        yok; kaynak makale (arXiv:2604.07759) GitHub deposunu
                        MIT gosteriyor ama VOC donusum betiginin kendi COCO
                        sablonu "Attribution License" (x-mol.com/groups/MIPC)
                        yaziyor. Kullanmadan once bizzat dogrulayin.

Kullanim
--------
    python tools/build_ship_dataset.py --data-dir "datasets" --out datasets/ship_merged --dry-run
    python tools/build_ship_dataset.py --data-dir "datasets" --out datasets/ship_merged --images-out datasets/ship_merged/images
"""

import argparse
import json
import re
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

try:  # `python tools/build_ship_dataset.py` dogrudan calistirilinca (kardes modul)
    from dataset_common import (
        Archive,
        Record,
        clip_box,
        extract_images,
        has_part,
        split_by_group_targets,
        session_key,
        swap_part,
    )
except ImportError:  # `tools.build_ship_dataset` olarak paket ustunden import edilince (testler)
    from tools.dataset_common import (
        Archive,
        Record,
        clip_box,
        extract_images,
        has_part,
        split_by_group_targets,
        session_key,
        swap_part,
    )

TARGET_NAMES = ("ship",)
SHIP = 1

#: Roboflow export'larindaki "root"/supercategory kalinti kategoriler hicbir
#: kutuda kullanilmiyor (id her zaman 0'a yakin, annotations'ta gecmiyor);
#: ama read_ship_coco() TUM bildirilen kategorileri esleme ister, yoksa
#: hata verir. Bu yuzden acikca None'a eslenmis olarak tabloda duruyorlar.
CLASS_MAPS = {
    "vais_smd_marvel": {
        "vessel": None,   # kullanilmayan root (VAIS/SMD/MARITIME/WSODD/MARVEL export'unun ust kategorisi)
        "buoy": None,     # arac degil, kasten hard-negative (VESSELimg'deki ayni karar)
        "object": SHIP,   # ⚠️ GERCEK gemi sinifi budur -- Roboflow kullanicisi boyle adlandirmis
    },
    "singapore_maritime": {
        "objects": None,          # kullanilmayan root
        "boat": SHIP,
        "buoy": None,             # hard-negative
        "ferry": SHIP,
        "flying bird-plane": None,  # alakasiz -- kus/ucak, gemi degil
        "kayak": SHIP,
        "other": None,            # belirsiz; ne oldugu dogrulanamadi, atildi
        "sail boat": SHIP,
        "speed boat": SHIP,
        "vessel-ship": SHIP,
    },
    "sea_vessels": {
        "sea-vessels": None,   # kullanilmayan root
        "fishing boat": SHIP,
        "merchant ship": SHIP,
        "military ship": SHIP,
        "patrol boat": SHIP,
        "sails boat": SHIP,
        "submarine": SHIP,
        "tugboat": SHIP,
        "yacht": SHIP,
    },
    "ship_model": {
        # Roboflow'un kendi "root" kategorisi de leaf kategorisi de "ship"
        # adini tasiyor (id'leri farkli, adlari ayni) -- ad bazli esleme
        # yaptigimiz icin tek satir ikisini de kapsiyor.
        "ship": SHIP,
    },
    "ir_thermal": {
        "boat": None,   # kullanilmayan root (0 kutu, gercek sinif adlari asagida)
        "bulk carrier": SHIP,
        "canoe": SHIP,
        "container ship": SHIP,
        "fishing boat": SHIP,
        "liner": SHIP,
        "sailboat": SHIP,
        "warship": SHIP,
    },
}

#: Kaynak -> (zip icindeki varsayilan dosya adi, okuyucu turu).
SOURCE_FILES = {
    "vais_smd_marvel": "VAIS_RGB-SMD-MARITIME-WSODD-MARVEL.v5-rgb_40_grayscale_60.coco.zip",
    "singapore_maritime": "Singapore maritime.v5i.coco.zip",
    "sea_vessels": "Sea Vessels Dataset.v2-sea_vessels_v2.coco.zip",
    "ship_model": "ship model.v4i.coco.zip",
    "ir_thermal": "ir.v1i.coco.zip",
    "wutdet": "WUTDet Part A.zip",
}

#: dedupe_sources() bu sirayla isler: zengin/adanmis kaynaklar once kimlik
#: "sahiplenir", vais_smd_marvel EN SONDA kalir ki zaten baskasinda olan
#: fotograflari/kareleri disarida biraksin. Sira, zenginlik olcumune dayanir:
#: singapore_maritime 9 alt-sinifli + Public Domain (SMD icin en zengin);
#: ship_model MARVEL-tarzi fotograflarin adanmis/daha buyuk kopyasi (renkli).
DEDUPE_ORDER = ("singapore_maritime", "ship_model", "sea_vessels",
                "ir_thermal", "wutdet", "vais_smd_marvel")

#: Hedef bolme: 70 / 15 / 15 (goruntu sayisi olarak; gruplar bolunmedigi
#: icin gerceklesen oran birkac puan sapabilir -- calisma ciktisi gercek
#: orani yazar). Roboflow'un KENDI train/valid/test bolmesi kullanilmaz:
#: kareleri rastgele bolmus, ayni oturum ucunde birden bulunuyor.
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
SPLIT_SEED = 1337


def cross_source_identity(file_name):
    """Farkli zip'lerde ayni fotografi/video karesini yakalayan anahtar.

    2026-08-11'de olculdu (bkz. modul basligi): MVI_XXXX_VIS_frameN video
    kare adlandirmasi (SMD kokenli) ve salt-numarali "<sayi>_jpg..." adlandirma
    (MARVEL kokenli) birden fazla kaynakta BIREBIR ayni ID'yle tekrarlaniyor.
    Baska bir adlandirma kalibina uymayan dosyalar icin None doner --
    o durumda cakisma kontrolu yapilmaz (yanlis pozitif sizintiden daha
    guvenli: bilinmeyen bir orintuyu es gecmek, olmayan bir cakismayi
    uydurmaktan iyidir).
    """
    base = file_name.rsplit("/", 1)[-1]
    # Video ID'si TEK BASINA yetmez -- ayni videonun onlarca farkli karesi
    # var; yalnizca "MVI_XXXX" yakalarsak o videonun TUM kareleri tek
    # kimlige collapse olur (2026-08-11'de olculdu: singapore_maritime
    # 6350 goruntuden 63'e dustu). Kare numarasina kadar eslenmeli.
    m = re.match(r"(MVI_\d+.*?_frame\d+)", base)
    if m:
        return "video:" + m.group(1)
    m = re.match(r"^(\d+)_jpg", base)
    if m:
        return "marvel:" + m.group(1)
    return None


#: SMD (Singapore Maritime Dataset) video karesi:
#: "MVI_1478_VIS_OB_frame90_jpg.rf.<hash>.jpg". Kare numarasi 'frame' ekine
#: YAPISIK oldugu icin genel session_key() bu kalibi goremiyor ve her kareyi
#: ayri oturum sayiyordu. OLCULDU (2026-08-12): 6350 goruntu -> 6350 "oturum",
#: yani grup bazli bolme rastgele bolmeye cokuyordu; 63 videonun 63'u de hem
#: train hem val'de cikti ve val karelerinin %93,6'sinin en yakin train karesi
#: 5 kare (~0,17 s) uzaktaydi. Kaynak oneki YOK: ayni video birden fazla
#: kaynakta bulunabiliyor (SMD hem singapore_maritime'da hem vais icinde).
SMD_VIDEO_RE = re.compile(r"^(MVI_\d+[A-Za-z_]*?)_frame\d+", re.IGNORECASE)

#: WUTDet Part A: 100K'lik setin her 20. karesi ("0000000.jpg", "0000020.jpg";
#: olculdu: 5024 ardisik farkin 5020'si tam 20). Makale kareleri 1-5 saniyede
#: bir ornekledigini soyluyor ve dizi bazli bolmeden hic bahsetmiyor
#: (arXiv:2604.07759). Ardisik ID'ler ayni sahne: olculdu (2026-08-12), 1068
#: val goruntusunun 994'unun ID komsusu train'deydi. Gercek video kimligi
#: dosya adinda YOK, o yuzden ID araligi yapay oturum olarak kullanilir.
WUTDET_ID_RE = re.compile(r"^(\d+)\.jpe?g$", re.IGNORECASE)
WUTDET_BUCKET_IDS = 1000          # 20'lik adimda ~50 goruntu

#: ir.v1i: dosya adlari yalnizca '1_XXXX' / '9_XXXX' iki on-eke sahip
#: (olculdu 2026-08-11: 8398 goruntunun 7398'i '1_', 1000'i '9_'), yani
#: session_key() TUM veriyi 2 deve gruba topluyordu ve bolme ya hepsini ya
#: hicbirini val'e atiyordu. On-ekin video kimligi mi yukleme sirasi mi
#: oldugu DOGRULANAMADI; temkinli taraf (gruplamayi kaldirmak degil,
#: inceltmek) secildi.
IR_THERMAL_RE = re.compile(r"^(\d+)_(\d+)_jpg")
IR_THERMAL_BUCKET = 200


def ship_session_key(file_name, source):
    """Bu 6 kaynagin adlandirmalarini bilen oturum anahtari.

    Genel session_key() Roboflow'un '<taban>_<kare>_jpg.rf.<hash>' kalibini
    tanir; asagidaki uc kaynak o kaliba UYMUYOR ve taninmadiklarinda her
    goruntu kendi oturumu olup bolme rastgeleye cokuyor. Tanimadigi bir ad
    icin genel isleve duser (sessizce yanlis gruplamaz).
    """
    base = file_name.rsplit("/", 1)[-1]

    match = SMD_VIDEO_RE.match(base)
    if match:
        return f"smd:{match.group(1).lower()}"

    if source == "wutdet":
        match = WUTDET_ID_RE.match(base)
        if match:
            return f"wutdet:{int(match.group(1)) // WUTDET_BUCKET_IDS}"

    if source == "ir_thermal":
        match = IR_THERMAL_RE.match(base)
        if match:
            return (f"ir_thermal:{match.group(1)}:"
                    f"{int(match.group(2)) // IR_THERMAL_BUCKET}")

    return session_key(file_name, source)


def dedupe_sources(by_source):
    """DEDUPE_ORDER sirasiyla isler; bir kimlik zaten alinmissa sonraki
    kaynaktaki kopyasini atar. Sozluk mutasyona ugratilir (kayitlar filtrelenir).

    Donus: {kaynak: atilan_kayit_sayisi} -- rapor icin.
    """
    claimed = set()
    dropped = {}
    for source in DEDUPE_ORDER:
        records = by_source.get(source, [])
        kept = []
        n_dropped = 0
        for rec in records:
            identity = cross_source_identity(rec.file_name)
            if identity is not None and identity in claimed:
                n_dropped += 1
                continue
            if identity is not None:
                claimed.add(identity)
            kept.append(rec)
        by_source[source] = kept
        dropped[source] = n_dropped
    return dropped


def read_ship_coco(archive, class_map, source, stats=None):
    """Standart Roboflow COCO export'u: train/valid/test + _annotations.coco.json.

    5 kaynagin 5'i de bu duzeni kullaniyor; tek okuyucu class_map ile
    parametrize edilip hepsine uygulanir (build_dataset.py'deki
    read_roboflow_coco'nun ayni deseni, ama LAND/SEA yerine tek sinif SHIP
    kullandigi ve "target=None -> kutuyu at, goruntuyu SAKLA" davranisi
    ayni kaldigi icin ayri fonksiyon: add_box() oradaki LAND/SEA'ye kilitli).
    """
    records = []
    members = sorted(n for n in archive.names()
                     if n.rsplit("/", 1)[-1] == "_annotations.coco.json")
    if not members:
        raise SystemExit(f"HATA: {source} icinde _annotations.coco.json bulunamadi")
    for member in members:
        base = member.rsplit("/", 1)[0] if "/" in member else ""
        data = json.loads(archive.read(member))
        cats = {c["id"]: str(c["name"]).strip().lower() for c in data["categories"]}
        unknown = set(cats.values()) - set(class_map)
        if unknown:
            raise SystemExit(
                f"HATA: {source} icinde eslenmemis kategori: {sorted(unknown)}. "
                f"tools/build_ship_dataset.py CLASS_MAPS tablosunu guncelleyin."
            )
        by_image = defaultdict(list)
        for ann in data["annotations"]:
            by_image[ann["image_id"]].append(ann)
        for image in data["images"]:
            member_img = f"{base}/{image['file_name']}" if base else image["file_name"]
            if not archive.exists(member_img):
                raise SystemExit(f"HATA: {source} goruntusu eksik: {member_img}")
            rec = Record(source, member_img, f"{source}/{image['file_name']}",
                        int(image["width"]), int(image["height"]), "train",
                        group=ship_session_key(image["file_name"], source))
            for ann in by_image.get(image["id"], ()):
                name = cats[ann["category_id"]]
                target = class_map[name]
                box = clip_box(*ann["bbox"], rec.width, rec.height)
                if box is None:
                    continue
                if target is None:
                    if stats is not None:
                        stats[f"{source}:drop:{name}"] += 1
                    continue
                rec.anns.append({"bbox": box, "category_id": SHIP,
                                 "iscrowd": 0, "ignore": 0})
                if stats is not None:
                    stats[f"{source}:keep:{name}"] += 1
            records.append(rec)
    return records


def read_wutdet_voc(archive, source="wutdet", stats=None):
    """WUTDet Part A: Pascal VOC XML (voc/Annotations/*.xml + voc/JPEGImages/).

    Resmi ImageSets/Main/{train,val}.txt bolmesine guvenilmiyor: bu projede
    incelenen her Roboflow kaynagi kare/kopya bazli sizinti tasiyordu (bkz.
    modul basligi); WUTDet'in kendi bolmesi de ayni riski tasiyabilir, hepsi
    "train" olarak okunup asagida oturum bazli yeniden bolunuyor.
    """
    records = []
    xml_members = sorted(n for n in archive.names()
                         if has_part(n, "Annotations") and n.lower().endswith(".xml"))
    if not xml_members:
        raise SystemExit(f"HATA: {source} icinde Annotations/*.xml bulunamadi")
    for member in xml_members:
        root = ET.fromstring(archive.read(member))
        filename = root.findtext("filename")
        if not filename:
            continue
        img_member = swap_part(member, "Annotations", "JPEGImages")
        img_member = img_member.rsplit("/", 1)[0] + "/" + filename
        if not archive.exists(img_member):
            raise SystemExit(f"HATA: {source} goruntusu eksik: {img_member}")
        size = root.find("size")
        width = int(size.findtext("width"))
        height = int(size.findtext("height"))
        rec = Record(source, img_member, f"{source}/{filename}", width, height,
                    "train", group=ship_session_key(filename, source))
        for obj in root.findall("object"):
            name = (obj.findtext("name") or "").strip().lower()
            bnd = obj.find("bndbox")
            if bnd is None:
                continue
            x1 = float(bnd.findtext("xmin"))
            y1 = float(bnd.findtext("ymin"))
            x2 = float(bnd.findtext("xmax"))
            y2 = float(bnd.findtext("ymax"))
            box = clip_box(x1, y1, x2 - x1, y2 - y1, width, height)
            if box is None:
                continue
            if name != "ship":
                if stats is not None:
                    stats[f"{source}:drop:{name}"] += 1
                continue
            rec.anns.append({"bbox": box, "category_id": SHIP,
                             "iscrowd": 0, "ignore": 0})
            if stats is not None:
                stats[f"{source}:keep:{name}"] += 1
        records.append(rec)
    return records


# --------------------------------------------------------------------------
# Dogrulama kapisi
# --------------------------------------------------------------------------
def validate(records):
    """Bozuk bir sey varsa dosya uretmeden hata verir.

    build_dataset.py'nin validate()'inden farki: VisDrone gibi 'resmi
    bolmede zaten sizinti var, kabul edilir' istisnasi yok -- burada hicbir
    kaynagin resmi bolmesine guvenilmiyor, dolayisiyla **hicbir oturum**
    birden fazla bolumde gorulmemeli (train/val/test tumu).
    """
    problems = []
    seen_files = set()
    groups = defaultdict(set)

    for r in records:
        if r.file_name in seen_files:
            problems.append(f"yinelenen dosya adi: {r.file_name}")
        seen_files.add(r.file_name)
        if r.width <= 0 or r.height <= 0:
            problems.append(f"gecersiz goruntu boyutu: {r.file_name}")
        groups[r.group].add(r.split)
        for ann in r.anns:
            x, y, w, h = ann["bbox"]
            if w <= 0 or h <= 0:
                problems.append(f"sifir/negatif kutu: {r.file_name} {ann['bbox']}")
            if x < -1e-6 or y < -1e-6 or x + w > r.width + 1e-6 \
                    or y + h > r.height + 1e-6:
                problems.append(f"sinir disi kutu: {r.file_name} {ann['bbox']} "
                                f"(goruntu {r.width}x{r.height})")
            if ann["category_id"] != SHIP:
                problems.append(f"gecersiz kategori: {ann['category_id']}")

    leaked = [g for g, s in groups.items() if len(s) > 1]
    if leaked:
        problems.append(
            f"{len(leaked)} oturum birden fazla bolumde "
            f"(ornek: {sorted(leaked)[:3]})"
        )
    return problems


def build_coco(records, split):
    subset = [r for r in records if r.split == split]
    images, annotations = [], []
    ann_id = 1
    for image_id, rec in enumerate(sorted(subset, key=lambda r: r.file_name), 1):
        images.append({
            "id": image_id,
            "file_name": rec.file_name,
            "width": rec.width,
            "height": rec.height,
            "source": rec.source,
        })
        for ann in rec.anns:
            annotations.append({
                "id": ann_id,
                "image_id": image_id,
                "category_id": ann["category_id"],
                "bbox": [round(v, 2) for v in ann["bbox"]],
                "area": round(ann["bbox"][2] * ann["bbox"][3], 2),
                "iscrowd": ann["iscrowd"],
            })
            ann_id += 1
    return {
        "info": {"description": "Gemi tespiti - tek sinifli COCO (6 kaynak birlesimi)",
                 "class_scheme": "ship"},
        "images": images,
        "annotations": annotations,
        "categories": [{"id": SHIP, "name": "ship", "supercategory": "vessel"}],
    }


def read_existing_ship_coco(out):
    """Daha once uretilmis ship_merged JSON'larini yeniden bolmek icin okur.

    Ham zip'ler yerelde artik bulunmasa bile, cikarilmis goruntuler ile COCO
    JSON'lari guvenli bir yeniden bolme icin yeterlidir. Oturum anahtari her
    zaman dosya adindan yeniden uretilir; eski JSON'un onceki (hatali) split
    bilgisine guvenilmez. Bu yol, *yalnizca* mevcut ciktiyi onarmak icindir;
    normal ilk olusturma yine ham kaynaklardan yapilir.
    """
    records = []
    seen_files = set()
    expected_categories = [{"id": SHIP, "name": "ship",
                            "supercategory": "vessel"}]

    for split in ("train", "val", "test"):
        path = out / "annotations" / f"instances_{split}.json"
        if not path.is_file():
            raise SystemExit(f"HATA: mevcut split bulunamadi: {path}")
        data = json.loads(path.read_text(encoding="utf-8"))
        if data.get("categories") != expected_categories:
            raise SystemExit(
                f"HATA: {path} tek sinifli ship COCO semasinda degil; "
                "yeniden bolme reddedildi."
            )

        images = data.get("images", [])
        by_id = {image.get("id"): image for image in images}
        if len(by_id) != len(images):
            raise SystemExit(f"HATA: {path} yinelenen image id iceriyor")
        anns_by_image = defaultdict(list)
        for ann in data.get("annotations", []):
            image_id = ann.get("image_id")
            if image_id not in by_id:
                raise SystemExit(f"HATA: {path} bilinmeyen image_id: {image_id}")
            if ann.get("category_id") != SHIP or len(ann.get("bbox", [])) != 4:
                raise SystemExit(f"HATA: {path} gecersiz ship anotasyonu: {ann}")
            anns_by_image[image_id].append(ann)

        for image in images:
            source = str(image.get("source", "")).strip()
            file_name = str(image.get("file_name", "")).replace("\\", "/")
            if source not in SOURCE_FILES:
                raise SystemExit(
                    f"HATA: {path} bilinmeyen/eksik kaynak etiketi: {source!r}"
                )
            if not file_name.startswith(f"{source}/"):
                raise SystemExit(
                    f"HATA: {path} kaynak ve dosya adi uyusmuyor: "
                    f"{source!r} / {file_name!r}"
                )
            if not file_name or file_name in seen_files:
                raise SystemExit(f"HATA: splitler arasi yinelenen dosya: {file_name}")
            seen_files.add(file_name)
            rec = Record(source, None, file_name, int(image.get("width", 0)),
                         int(image.get("height", 0)), split,
                         group=ship_session_key(file_name, source))
            for ann in anns_by_image[image["id"]]:
                rec.anns.append({
                    "bbox": ann["bbox"], "category_id": SHIP,
                    "iscrowd": int(ann.get("iscrowd", 0)), "ignore": 0,
                })
            records.append(rec)
    return records


def repartition_existing_ship_coco(out, val_fraction, test_fraction, dry_run):
    """Mevcut ship_merged ciktilarini oturum bazli %70/%15/%15 yeniden boler."""
    records = read_existing_ship_coco(out)

    # Bir video/oturum iki kaynakta kalmissa kaynak-bazli kota atamasi onu
    # farkli splitlere koyabilir. Ham kaynaklardan global politika ile yeniden
    # uretmek gerekir; burada sessizce sizinti yaratmak yerine duruyoruz.
    group_sources = defaultdict(set)
    for record in records:
        group_sources[record.group].add(record.source)
    shared = sorted(group for group, sources in group_sources.items()
                    if len(sources) > 1)
    if shared:
        raise SystemExit(
            "HATA: %d oturum birden fazla kaynakta goruluyor "
            "(ornek: %s). Ham zip'lerden global yeniden bolme gerekir."
            % (len(shared), shared[:3])
        )

    by_source = defaultdict(list)
    for record in records:
        by_source[record.source].append(record)
    for source in DEDUPE_ORDER:
        source_records = by_source.get(source, [])
        if not source_records:
            raise SystemExit(f"HATA: mevcut cikti icinde kaynak yok: {source}")
        counts = split_by_group_targets(source_records, val_fraction,
                                        test_fraction, seed=SPLIT_SEED)
        print(f"   {source:<20}{len(source_records):>7} goruntu, "
              f"{len({r.group for r in source_records}):>6} oturum -> "
              f"train {counts.get('train', 0)} / val {counts.get('val', 0)} / "
              f"test {counts.get('test', 0)}")

    problems = validate(records)
    if problems:
        print("\n" + "!" * 74)
        print(f"DOGRULAMA BASARISIZ - {len(problems)} sorun")
        for problem in problems[:20]:
            print("  -", problem)
        raise SystemExit(1)

    splits = [(name, build_coco(records, name))
              for name in ("train", "val", "test")]
    print("\nDogrulama gecti: oturum sizintisi yok, kutular ve tek sinif semasi gecerli.")
    print(f"{'':<8}{'goruntu':>10}{'kutu':>10}{'bos':>8}")
    for name, coco in splits:
        with_box = {ann["image_id"] for ann in coco["annotations"]}
        print(f"{name:<8}{len(coco['images']):>10}{len(coco['annotations']):>10}"
              f"{len(coco['images']) - len(with_box):>8}")

    if dry_run:
        print("\n--dry-run: mevcut JSON'lar degistirilmedi.")
        return

    # Tum JSON'lar once bellekte kuruldu ve denetlendi. Gecici dosyalar
    # tamamlanmadan mevcut manifestin uzerine yazilmaz; yarim bolme kalmaz.
    targets = []
    for name, coco in splits:
        target = out / "annotations" / f"instances_{name}.json"
        temporary = target.with_suffix(".json.tmp")
        temporary.write_text(json.dumps(coco), encoding="utf-8")
        targets.append((temporary, target))
    for temporary, target in targets:
        temporary.replace(target)
    print(f"\nyazildi: {out / 'annotations'} (oturum-bazli yeniden bolme)")


def summarise(by_source_final, stats, dropped_by_dedupe):
    print("\n" + "=" * 74)
    print("KAYNAK BAZLI OZET (dedupe SONRASI)")
    print("=" * 74)
    header = f"{'kaynak':<20}{'goruntu':>9}{'train':>8}{'val':>7}{'test':>7}{'kutu':>9}{'bos':>7}"
    print(header); print("-" * len(header))
    grand_img = grand_box = 0
    for source in DEDUPE_ORDER:
        recs = by_source_final.get(source, [])
        n_img = len(recs)
        n_train = sum(1 for r in recs if r.split == "train")
        n_val = sum(1 for r in recs if r.split == "val")
        n_test = sum(1 for r in recs if r.split == "test")
        n_box = sum(len(r.anns) for r in recs)
        n_empty = sum(1 for r in recs if not r.anns)
        grand_img += n_img; grand_box += n_box
        print(f"{source:<20}{n_img:>9}{n_train:>8}{n_val:>7}{n_test:>7}{n_box:>9}{n_empty:>7}")
    print("-" * len(header))
    print(f"{'TOPLAM':<20}{grand_img:>9}{'':>8}{'':>7}{'':>7}{grand_box:>9}")

    print("\nCAKISMA NEDENIYLE ATILAN (dedupe_sources)")
    for source in DEDUPE_ORDER:
        n = dropped_by_dedupe.get(source, 0)
        if n:
            print(f"  {source:<20}{n:>6} goruntu (baska kaynakta zaten var)")

    print("\nSINIF ESLEME DOKUMU (kaynak:karar:orijinal-ad)")
    for key in sorted(stats):
        print(f"  {key:<45} {stats[key]:>8}")


def main():
    parser = argparse.ArgumentParser(
        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir", default="datasets")
    parser.add_argument("--out", default="datasets/ship_merged")
    parser.add_argument("--val-fraction", type=float, default=VAL_FRACTION)
    parser.add_argument("--test-fraction", type=float, default=TEST_FRACTION,
                        help="kaynak-bazli ayrilacak test orani")
    parser.add_argument("--images-out", default=None)
    parser.add_argument("--dry-run", action="store_true")
    parser.add_argument("--repartition-existing", action="store_true",
                        help="mevcut --out/annotations JSON'larini ham zip "
                        "olmadan oturum bazli yeniden bol")
    parser.add_argument("--source", action="append", default=None,
                        metavar="ETIKET=YOL",
                        help="tek bir kaynagin yolunu degistir; birden fazla "
                             "kez verilebilir")
    parser.add_argument("--skip", action="append", default=None,
                        metavar="ETIKET",
                        help="bu kaynagi tamamen atla (lisans netlesene kadar "
                             "orn. --skip wutdet)")
    args = parser.parse_args()

    if not 0 < args.val_fraction < 1 or not 0 <= args.test_fraction < 1 \
            or args.val_fraction + args.test_fraction >= 1:
        raise SystemExit(
            "HATA: val orani pozitif, test orani negatif olmayan olmali; "
            "toplamlari 1'den kucuk olmali"
        )

    if args.repartition_existing:
        if args.test_fraction == 0:
            raise SystemExit("HATA: --repartition-existing icin test orani pozitif olmali")
        repartition_existing_ship_coco(Path(args.out), args.val_fraction,
                                       args.test_fraction, args.dry_run)
        return

    data_dir = Path(args.data_dir)
    skip = set(args.skip or ())
    overrides = {}
    for item in (args.source or ()):
        if "=" not in item:
            raise SystemExit(f"HATA: 'etiket=yol' bekleniyordu: {item}")
        key, value = item.split("=", 1)
        overrides[key.strip()] = Path(value.strip())
    unknown = set(overrides) - set(SOURCE_FILES)
    if unknown:
        raise SystemExit(f"HATA: bilinmeyen kaynak etiketi: {sorted(unknown)}")

    stats = Counter()
    by_source = {}
    for source, filename in SOURCE_FILES.items():
        if source in skip:
            print(f">> {source:<20} ATLANDI (--skip)")
            continue
        path = overrides.get(source, data_dir / filename)
        if not path.exists():
            raise SystemExit(f"HATA: kaynak bulunamadi: {path}")
        print(f">> {source:<20} {path.name}")
        with Archive(path) as archive:
            if source == "wutdet":
                got = read_wutdet_voc(archive, source=source, stats=stats)
            else:
                got = read_ship_coco(archive, CLASS_MAPS[source], source, stats=stats)
        if not got:
            raise SystemExit(f"HATA: '{source}' kaynagindan hic goruntu okunamadi")
        for rec in got:
            rec.origin = path
        by_source[source] = got

    # Kopya eleme BOLMEDEN once: sonra elenirse bolme oranlari bozulur
    # (olculdu 2026-08-12: vais_smd_marvel bolundukten sonra 24.648'den
    # 14.421'e dusunce val payi hedef %25 yerine %10,6 cikmisti).
    dropped = dedupe_sources(by_source)

    # Her kaynak AYRI bolunur ki kucuk kaynaklar (sea_vessels, 736 goruntu)
    # buyuklerin yaninda kaybolmasin; global karistirma boyuta gore
    # agirliklanirdi. Gruplar bolunmez -> oturum sizintisi olmaz.
    for source, recs in by_source.items():
        counts = split_by_group_targets(recs, args.val_fraction,
                                        args.test_fraction, seed=SPLIT_SEED)
        n = len(recs)
        groups = len({r.group for r in recs})
        print(f"   {source:<20}{n:>7} goruntu, {groups:>6} oturum -> "
              f"train {counts.get('train',0)} / val {counts.get('val',0)} / "
              f"test {counts.get('test',0)}")

    records = [r for recs in by_source.values() for r in recs]

    problems = validate(records)
    if problems:
        print("\n" + "!" * 74)
        print(f"DOGRULAMA BASARISIZ - {len(problems)} sorun")
        for p in problems[:20]:
            print("  -", p)
        if len(problems) > 20:
            print(f"  ... ve {len(problems)-20} tane daha")
        raise SystemExit(1)
    print("\nDogrulama gecti: kutular sinir icinde, kategori {1}, "
          "oturum sizintisi yok, kaynaklar arasi kopya yok.")

    summarise(by_source, stats, dropped)

    splits = [(name, build_coco(records, name)) for name in ("train", "val", "test")]
    print("\n" + "=" * 74)
    print(f"{'':<8}{'goruntu':>10}{'kutu':>10}{'bos':>8}")
    for name, coco in splits:
        real = [a for a in coco["annotations"]]
        with_box = {a["image_id"] for a in real}
        empty = len(coco["images"]) - len(with_box)
        print(f"{name:<8}{len(coco['images']):>10}{len(real):>10}{empty:>8}")

    if args.dry_run:
        print("\n--dry-run: dosya yazilmadi.")
        return

    out = Path(args.out)
    (out / "annotations").mkdir(parents=True, exist_ok=True)
    for name, coco in splits:
        target = out / "annotations" / f"instances_{name}.json"
        target.write_text(json.dumps(coco), encoding="utf-8")
        print(f"yazildi: {target}")

    manifest = out / "image_manifest.json"
    manifest.write_text(json.dumps({
        "sources": {s: str(SOURCE_FILES[s]) for s in by_source},
        "licenses": {
            "vais_smd_marvel": "CC BY 4.0",
            "singapore_maritime": "Public Domain",
            "sea_vessels": "CC BY 4.0",
            "ship_model": "CC BY 4.0",
            "ir_thermal": "CC BY 4.0",
            "wutdet": "BELIRSIZ - kullanmadan once dogrulayin (bkz. modul basligi)",
        },
        "members": [{"source": r.source, "member": r.member,
                     "file_name": r.file_name} for r in records],
    }), encoding="utf-8")
    print(f"yazildi: {manifest}  (goruntuleri cikarmak icin)")

    if args.images_out:
        extract_images(records, Path(args.images_out))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile yolox_tiny_ship.py
#!/usr/bin/env python3
"""KV260 DPU'suna uyumlu YOLOX-Tiny -- tek sinif "ship" (gemi tespiti pivotu).

Bu, havadan land+sea 2 sinifli projeden AYRI bir dal: kamera artik gemiyle
yaklasik ayni mesafede/yukseklikte (kiyi, iskele, gemi guvertesi), yukaridan
degil. Veri `tools/build_ship_dataset.py` ile 6 kaynaktan birlestirildi
(bkz. o dosyanin basligi -- kaynaklar arasi 2 buyuk kopya kumesi olculup
elendi). Mimari ve DPU uyarlamalari (ReLU + DPUFocus), aerial projede
kartta dogrulanmis Tiny geometrisinden **degistirilmeden** devralinir --
tek degisken veri ve girdi boyutu olsun diye.

Girdi boyutu -- 512x512, olculerek secildi (2026-08-11)
---------------------------------------------------------
Aerial projenin 896x512 karari VisDrone'un neredeyse tamami 16:9/4:3 oldugu
ve kutularin uniform kucuk oldugu bir veri icindi. Bu veri seti FARKLI:
kaynaklarin **%62'si kare** (sea_vessels, ship_model, vais_smd_marvel hepsi
1:1 kirpilmis/stretch'lenmis), yalnizca %22'si 16:9 (singapore_maritime,
wutdet). Kare kanvas bu yuzden burada letterbox israfini aerial'daki
mantigin TERSINE cevirerek en aza indiriyor.

Kutu boyutu da heterojen (aerial'daki gibi uniform kucuk degil): p10=%2.1,
medyan=%8.0, p90=%80.5 (goruntu genisligine oran). 512x512'de olculdu:

    cand=384   p10=8.1px  p25=13.8px  <8px: %9.6   <16px: %29.6
    cand=448   p10=9.4px  p25=16.1px  <8px: %6.5   <16px: %24.9
    cand=512   p10=10.8px p25=18.4px  <8px: %4.2   <16px: %20.4
    cand=640   p10=13.5px p25=23.0px  <8px: %2.3   <16px: %14.1

512 secildi: FPS bu pivotun ana gerekcesi (bkz. HANDOFF_CLAUDE.md §5 --
kartta olculen darboğaz DPU degil, ARM CPU'daki on-isleme; on-isleme suresi
piksel sayisiyla dogru orantili). 512x512 = 262.144 px, 896x512'nin (aerial)
%57'si -- DPU ve on-isleme sureleri kabaca ayni oranda dusmesi beklenir
(TAHMIN, kartta olculmedi). Karsiliginda kutularin yalnizca %4,2'si 8px
altina dusuyor (aerial'da VisDrone'un tamami zaten kucuktu, secim yoktu;
burada VAR ve FPS lehine kullanildi). 640 FPS'te daha az kazandirir ama
kucuk nesne kaybi da azalir -- dogruluk yetersiz kalirsa once denenecek
adim budur, mimariyi degistirmeye gerek yok.

DPU uyumlulugu (aerial projeden degismeden devralindi)
---------------------------------------------------------
  1. act="relu": DPU, SiLU'yu desteklemez.
  2. DPUFocus: Focus'un strided-slice'i DPU'da calismaz; sabit agirlikli
     conv ayni space-to-depth'i uretir.
  3. depthwise=False (Tiny geometrisi): Nano'nun depthwise conv'lari
     per-tensor INT8'de coktu (bkz. HANDOFF §5); Tiny bu sorunu tasimiyor.

Alan-saglamligi augmentasyonu -- termal/gri ton/parlama, arastirilarak eklendi
---------------------------------------------------------------------------------
Karta gidecek gercek girdi buyuk ihtimalle RGB olmayacak (termal kamera veya
gri ton video akisi) ve deniz yuzeyi gunes yansimasi/parlama gibi bozulmalar
tasiyacak. Egitim verisi bunun bir kismini zaten dogal olarak icin
(ir_thermal: gercek termal; vais_smd_marvel'in %60'i gri tona cevrilmis) ama
RGB kaynaklarin (ship_model, sea_vessels, singapore_maritime'in cogu) modelin
renge fazla guvenmesine yol acma riski var. Cozum, egitim sirasinda RGB
goruntuleri de olasiliklarla ayni bozulmalara maruz birakmak.

Kaynak arastirmasi (2026-08-12):
  * Albumentations (Buslaev ve ark. 2020, Information dergisi, MDPI --
    hakemli, https://www.mdpi.com/2078-2489/11/2/125) bu tur pikselsel
    augmentasyonlar icin standart, gecerliligi olculmus kutuphane.
  * ToGray: YOLOv8 tabanli bir dusme-tespiti calismasinda dusuk olasilikla
    uygulanip modelin rengi degil parlaklik/sekli kullanmayi ogrenmesini
    sagladigi raporlandi -- burada TAM olarak istedigimiz ozellik.
  * CLAHE (Contrast Limited Adaptive Histogram Equalization): BVLOS drone
    engel-tespiti calismasinda test edilen 8 Albumentations tekniginden
    biri; dusuk kontrastli (termal) ve asiri-pozlanmis (parlama) goruntuleri
    ayni ailede ele alan klasik, kanitlanmis bir teknik.
  * RandomBrightnessContrast/parlama: "Enhancing Maritime Object Detection
    ... with Data Augmentation" (arXiv:2510.07346) parlaklik/kontrast
    augmentasyonunun DENIZCILIK goruntulerinde olculmus faydasini raporluyor
    (birlesik pipeline ile mAP@0.5 0.80 -> 0.89, +0.09). Bu makale gunes
    parlamasini/yansimayi acikca "kacinilmasi gereken saptirici" olarak
    tanimliyor -- bizim RandomSunFlare/RandomShadow eklentimizin gerekcesi.
  * GaussNoise/ISONoise: dusuk isikli/termal sensorlerin taninan gurultu
    profili; ir.v1i orneklerinde gozle de gorulen tane/gurultu var.

Her teknik dusuk-orta olasilikla (%15-30) uygulanir: amac goruntuleri hep
bozmak degil, egitim boyunca CESITLI kosullara maruz birakmaktir. Yalnizca
egitimde (get_dataset); dogrulama (get_eval_dataset) bozulmamis kalir ki
AP sayisi augmentasyon sansina bagli olmasin.

Gri/termal icin ikinci katman -- zincirin SONUNDA (GrayThermalTransform)
------------------------------------------------------------------------
Yukaridaki ToGray tek basina yetmiyor: zincirin basinda, YOLOX'un
augment_hsv'sinden ONCE calisiyor ve augment_hsv doygunlugu carpmiyor
TOPLUYOR (S=0'a +30'a kadar ekleyebiliyor). Olculdu (2026-08-12, YOLOX
6ddff48'deki fonksiyonun birebir kopyasiyla, 200 deneme): gri goruntulerin
%13'u yeniden renklendi. Bu yuzden gri'ye cevirme bir kez de TUM zincirin
sonunda yapilir; oraya hicbir sey dokunamaz. Ayni yerde, gri orneklerin bir
kisminin polaritesi ters cevrilir (termal white-hot / black-hot -- kamera
ayari, veri ozelligi degil).

Dayanikliligi OLCMEK icin (varsaymak yerine):
    SHIP_EVAL_GRAY=1 python YOLOX/tools/eval.py -f yolox_tiny_ship.py ...
ayni checkpoint'i gri'ye cevrilmis val uzerinde olcer; AP dususu
dayanikliligin sayisal karsiligidir. Gercek termal icin kaynak bazli AP'ye
bakin (ir_thermal) -- bkz. not defterindeki kaynak-bazli degerlendirme.
"""

import os
import random
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

for _alias, _type in (("float", float), ("int", int), ("bool", bool)):
    if _alias not in np.__dict__:
        setattr(np, _alias, _type)

from yolox.data import COCODataset
from yolox.exp import Exp as MyExp

for _helper_dir in (Path(__file__).resolve().parent,
                    Path(__file__).resolve().parent.parent):
    if str(_helper_dir) not in sys.path:
        sys.path.insert(0, str(_helper_dir))

#: Saha tanimi: tum gemi/tekne tipleri tek sinifta (tools/build_ship_dataset.py).
TARGET_CLASSES = ("ship",)


def assert_class_scheme(coco, expected):
    """Anotasyon semasi modelin sinif sayisiyla uyusmuyorsa hemen durur.

    Eski bir sema (orn. 2 sinifli aerial `instances_*.json`) diskte kalirsa
    model sessizce yanlis etiketlerle egitilir. Ucuz bir kapi, pahali bir
    hatayi onler (aerial exp'inde ayni desen).
    """
    if expected is None:
        return
    found = sorted(coco.cats)
    if found != list(range(1, expected + 1)):
        names = [coco.cats[c].get("name", c) for c in found]
        raise ValueError(
            f"Anotasyon semasi uyusmuyor: {expected} sinif bekleniyordu, "
            f"{len(found)} bulundu ({names}). Veriyi "
            f"'tools/build_ship_dataset.py' ile yeniden uretin."
        )


def _build_robust_augmentor():
    """RGB goruntuleri termal/gri-ton/parlama kosullarina maruz birakan
    pikselsel augmentasyon zinciri. Gerekce ve kaynaklar modul basliginda.

    Hepsi PIKSELSEL (kutu koordinatlarini degistirmez) -- bbox_params
    gerekmiyor. load_resized_img() asamasinda, goruntu 512'ye kucultuldukten
    SONRA ve YOLOX'un kendi TrainTransform'undan ONCE calisip duz bir numpy
    goruntu doner (neden kucultmeden sonra: bkz. RobustShipDataset).
    """
    try:
        import albumentations as A
    except ImportError as exc:
        raise SystemExit(
            "HATA: albumentations kurulu degil. Kaggle imajinda varsayilan "
            "olarak bulunur; yerelde 'pip install albumentations' calistirin."
        ) from exc

    return A.Compose([
        # Renge degil parlaklik/sekle guvenmeyi ogretir -- termal/gri ton
        # girdiyle dogrudan ayni dagilim (bkz. modul basligi, ToGray).
        A.ToGray(p=0.25),
        # Deniz yuzeyi gunes yansimasi/parlama VE dusuk kontrastli termal --
        # ayni ailede, tek seferde birini uygular (OneOf: gercekci kalsin
        # diye ustuste yiginlanmaz).
        A.OneOf([
            A.RandomSunFlare(src_radius=150, p=1.0),
            A.RandomShadow(p=1.0),
            A.CLAHE(clip_limit=3.0, p=1.0),
        ], p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.3),
        # Dusuk isik/termal sensor gurultusu.
        A.OneOf([
            A.GaussNoise(p=1.0),
            A.ISONoise(p=1.0),
        ], p=0.15),
    ])


#: Aga giren goruntunun GARANTILI gri olma olasiligi. Neden ayri bir adim,
#: neden Albumentations'taki ToGray yetmiyor: YOLOX'un augment_hsv'si
#: (data_augment.py) doygunlugu CARPMIYOR, TOPLUYOR -- S=0 olan gri bir
#: goruntuye +30'a kadar doygunluk ekleyebiliyor. Olculdu (2026-08-12, YOLOX
#: 6ddff48'deki fonksiyonun birebir kopyasiyla, 200 deneme): gri goruntulerin
#: %13'u yeniden renklendi, en buyuk kanal farki 29/255. ToGray load_image'ta
#: (HSV'den ONCE) calistigi icin bu geri alma kacinilmaz; bu yuzden gri'ye
#: cevirme burada, TUM augmentasyon zincirinin SONUNDA bir kez daha yapilir.
TRAIN_GRAY_PROB = 0.40

#: Gri'ye cevrilen orneklerin icinde polaritesi ters cevrilenlerin orani.
#: Termal kameralarda white-hot / black-hot iki ayri gorunum uretir ve bu
#: KAMERA AYARIDIR -- sahada hangisinin secilecegi bilinmiyor. Yalnizca
#: white-hot ile egitilen model digerinde coker. Literaturde termal veri
#: uretiminde "intensity inversion" kullaniliyor (bkz. Liu ve ark. 2021,
#: Mobile Information Systems -- CycleGAN + yogunluk tersleme).
#: DIKKAT: bu bir TAHMIN, bu veri setinde faydasi olculmedi; egitim sonrasi
#: kaynak bazli AP ile (ozellikle ir_thermal) dogrulanmali.
TRAIN_INVERT_PROB = 0.25

#: BT.601 parlaklik agirliklari. YOLOX preproc'u kanal sirasini BGR birakir
#: (RGB'ye cevirmez) ve CHW dondurur -- dogrulandi: preproc() icinde yalnizca
#: transpose(2,0,1) var, [:, :, ::-1] yok.
_BGR_LUMA = np.array([0.114, 0.587, 0.299], dtype=np.float32).reshape(3, 1, 1)


class GrayThermalTransform:
    """Bir YOLOX preproc'unu sarar; CIKTIYI gri (ve istege bagli ters) yapar.

    Zincirin en sonunda durur: MosaicDetection -> TrainTransform -> augment_hsv
    -> flip -> BURASI. Boylece hicbir sonraki adim griligi bozamaz.

    Girdi/cikti sozlesmesi YOLOX'unkiyle ayni: preproc CHW float32, BGR,
    0-255 araliginda goruntu dondurur (normalize etmez).
    """

    def __init__(self, inner, gray_prob, invert_prob=0.0):
        self.inner = inner
        self.gray_prob = gray_prob
        self.invert_prob = invert_prob

    def __call__(self, image, target, input_dim):
        image, target = self.inner(image, target, input_dim)
        if random.random() < self.gray_prob:
            luma = (image * _BGR_LUMA).sum(axis=0, keepdims=True)
            image = np.repeat(luma, 3, axis=0)
            if random.random() < self.invert_prob:
                # Letterbox dolgusu (114) da terslenir -> 141. Kabul edildi:
                # dolgu her iki halde de duz bir alan, ve model her iki
                # degeri de gorerek daha saglam olur.
                image = 255.0 - image
        return np.ascontiguousarray(image, dtype=np.float32), target


class RobustShipDataset(COCODataset):
    """COCODataset + load_image() asamasinda alan-saglamligi augmentasyonu.
    Yalnizca EGITIMDE kullanilir (get_dataset); dogrulama (COCODataset,
    duz) bozulmamis kalir ki AP sayisi augmentasyon sansina bagli olmasin.

    Tek override load_resized_img(); geri kalan COCODataset davranisina
    dokunulmaz.
    """

    _augmentor = None  # ilk ornekte kurulur, sonra tekrar kurulmaz

    def __init__(self, *args, expected_num_classes=None, **kwargs):
        # cache=True COCODataset'in read_img()'ini RAM/disk onbellegine
        # yazar; augmentasyon zincirimiz o cagrinin altinda kaldigi icin ilk
        # epoch'ta DONULUR ve her epoch ayni bozulma gorulur. Reddet.
        if kwargs.get("cache", False):
            raise ValueError(
                "RobustShipDataset ile --cache kullanmayin: augmentasyon "
                "ilk epoch'ta donar, her epoch farkli bozulma gormez."
            )
        super().__init__(*args, **kwargs)
        assert_class_scheme(self.coco, expected_num_classes)

    def load_resized_img(self, index):
        """Augmentasyon KUCULTMEDEN SONRA calisir -- olculerek buraya tasindi.

        load_image()'i sarmak dogal duruyordu ama orada goruntu HAM
        cozunurluktedir. Kaggle'da olculdu (2026-08-11 kosusu, epoch 1-2):
        iter_time 1,33 s'nin 0,99 s'si veri bekleme, yani GPU'nun %73'u bos.
        Zincirin maliyeti piksel sayisiyla dogru orantili:
        1920x1080'de 66 ms, 650x650'de 17 ms, 512x512'de 9 ms -- ve egitim
        setinin %25'i 1920x1080. Mozaik yuzunden ornek basina ~2,5 cagri.

        load_resized_img() ayni goruntuyu 512'ye sigacak sekilde kucultulmus
        halde verir; augmentasyon ayni transformlar ve ayni olasiliklarla
        orada calisinca egitim icerigi degismez, maliyet 2-7 kat duser.
        Yan fayda: RandomSunFlare(src_radius=150) artik gercekten gorunur bir
        parlama uretir; ham 1920px'te uygulanip 512'ye inince ~40px'e
        buzuluyordu.

        Enjeksiyon noktasi olarak read_img() degil bu secildi: read_img
        onbellek dekoratorlu (cache_read_img), oraya yazmak --cache acikken
        augmentasyonu dondururdu. __init__ zaten cache'i reddediyor ama
        savunma tek noktaya yigilmasin.
        """
        img = super().load_resized_img(index)
        cls = type(self)
        if cls._augmentor is None:
            cls._augmentor = _build_robust_augmentor()
        return cls._augmentor(image=img)["image"]


class DPUFocus(nn.Module):
    """YOLOX Focus katmaninin DPU-uyumlu birebir karsiligi.

    (aerial exp'indeki DPUFocus ile birebir ayni; kartta dogrulandi --
     2026-08-10, golden test ~1e-5 px farkla gecti.)
    """

    def __init__(self, in_channels, out_channels, ksize=1, stride=1, act="silu"):
        super().__init__()
        from yolox.models.network_blocks import BaseConv

        self.space_to_depth = nn.Conv2d(
            in_channels, in_channels * 4, kernel_size=2, stride=2, bias=False
        )
        w = torch.zeros(in_channels * 4, in_channels, 2, 2)
        for block, (r, c) in enumerate(((0, 0), (1, 0), (0, 1), (1, 1))):
            for ch in range(in_channels):
                w[block * in_channels + ch, ch, r, c] = 1.0
        with torch.no_grad():
            self.space_to_depth.weight.copy_(w)
        self.space_to_depth.weight.requires_grad = False
        self.conv = BaseConv(in_channels * 4, out_channels, ksize, stride, act=act)

    def forward(self, x):
        return self.conv(self.space_to_depth(x))


class Exp(MyExp):
    def __init__(self):
        super().__init__()
        # ---- model: YOLOX-Tiny geometrisi (resmi exps/default/yolox_tiny.py) ----
        self.depth = 0.33
        self.width = 0.375
        self.act = "relu"
        self.depthwise = False
        self.num_classes = len(TARGET_CLASSES)

        # ---- girdi boyutu: olculerek secildi, bkz. modul basligi ----
        self.input_size = (512, 512)
        self.test_size = (512, 512)
        # random_resize: 32*s her iki eksende de (kare kanvas), s in [12, 18]
        # -> 384x384 .. 576x576 araligi.
        self.random_size = (12, 18)

        # ---- veri seti ----
        # tools/build_ship_dataset.py ciktisi: annotations/ + images/<kaynak>/<ad>.jpg
        self.data_dir = "datasets/ship_merged"
        self.train_ann = "instances_train.json"
        # Test seti yalnizca EN SONDA, bir kez olculmeli; ayri bir exp
        # dosyasi kopyalamak yerine ortam degiskeniyle secilir:
        #   SHIP_EVAL_ANN=instances_test.json python YOLOX/tools/eval.py ...
        self.val_ann = os.environ.get("SHIP_EVAL_ANN", "instances_val.json")
        # Gri/termal dayanikliligi OLCMEK icin: ayni checkpoint, gri'ye
        # cevrilmis val. AP dususu dayanikliligin sayisal karsiligidir.
        #   SHIP_EVAL_GRAY=1 ...
        self.eval_gray = os.environ.get("SHIP_EVAL_GRAY", "") == "1"
        self.image_folder = "images"
        # 2 iken veri hatti darbogazdi (olculdu: data_time 0,99 s / iter_time
        # 1,33 s). Kaggle GPU imajlari 4 vCPU verir; augmentasyon artik
        # kucultulmus goruntude calistigi icin 4 worker GPU'yu doyurmali.
        self.data_num_workers = 4
        self.dataset = None

        # ---- augmentasyon ----
        # Kutu basi ortalama nesne sayisi dusuk (92920 kutu / 39647 goruntu
        # ~= 2,3), ama kumelenmis sahneler var (bir ir_thermal marina karesi
        # 36 kutu tasiyordu) -- YOLOX varsayilani (50) yetersiz kalabilir,
        # aerial'in asiri degeri (1000/4000) ise burada gereksiz.
        self.mosaic_prob = 0.5
        self.mosaic_scale = (0.5, 1.5)
        self.enable_mixup = False

        # ---- egitim ----
        # Bu veri setinde epoch-vs-AP egrisi HENUZ OLCULMEDI (aerial'da
        # ~epoch 25'te doymustu, ama o farkli bir veri seti/gorevdi).
        # 30 epoch temkinli bir baslangic; ilk egitimde AP@0.50 egrisine
        # bakip gerekirse kisaltin/uzatin -- varsayim degil, olcum.
        self.max_epoch = 30
        # YOLOX varsayilani 5'tir ama o 300 epoch icindir (%1,7). Burada 30
        # epoch var; 5 warmup + 8 no_aug birakinca tam augmentasyonlu pencere
        # 17 epoch'a iniyordu. Ustelik bu bir SIFIRDAN egitim degil, COCO
        # agirliklarindan fine-tune -- katmanlar zaten makul bir noktada,
        # uzun LR rampasina ihtiyac yok. 2 ile pencere 20 epoch'a cikiyor.
        # Ilk epoch'larda loss ziplarsa (onceki kosuda 12,2'den duzgun
        # inmisti) once buraya bakin.
        self.warmup_epochs = 2
        self.no_aug_epochs = 8
        self.eval_interval = 5
        self.print_interval = 50
        self.save_history_ckpt = False  # Kaggle diskini doldurmamak icin

        # ---- kartin calisma noktasi ----
        # Kartta calisan uygulama sabit bir guven esigi kullaniyor
        # (deploy/src/main.cpp:164 -> conf_thr = 0.15f). AP bu esikten
        # bagimsizdir ama saha sorusu ("kac gemiyi kaciriyoruz, kac yanlis
        # alarm") esige baglidir; quantize_yolox.py P/R/F1'i bu degerde
        # raporlar. Buradaki sayi main.cpp'deki ile AYNI kalmali, yoksa
        # olctugumuz calisma noktasi kartinkinden farkli olur.
        self.deploy_conf = 0.15

        # Tekrarlanabilirlik: YOLOX train.py bunu gorurse random/torch
        # seed'lenir ve cudnn.deterministic acilir (YOLOX'un kendi uyarisi:
        # egitimi yavaslatabilir). Seed'siz kosuda ayni kod iki farkli
        # sonuc verir ve "degisiklik ise yaradi mi" sorusu cevaplanamaz.
        # Hiz gerekirse None yapin -- ama o zaman sonuclari kiyaslamayin.
        self.seed = 1337

        self.exp_name = os.path.split(os.path.realpath(__file__))[1].split(".")[0]

    def get_model(self, sublinear=False):
        def init_yolo(M):
            for m in M.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eps = 1e-3
                    m.momentum = 0.03

        if "model" not in self.__dict__:
            from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead

            in_channels = [256, 512, 1024]
            backbone = YOLOPAFPN(
                self.depth, self.width, in_channels=in_channels,
                act=self.act, depthwise=self.depthwise,
            )
            head = YOLOXHead(
                self.num_classes, self.width, in_channels=in_channels,
                act=self.act, depthwise=self.depthwise,
            )
            self.model = YOLOX(backbone, head)
            self.model.backbone.backbone.stem = DPUFocus(
                3, int(self.width * 64), ksize=3, act=self.act
            )

        self.model.apply(init_yolo)
        self.model.head.initialize_biases(1e-2)
        return self.model

    def get_dataset(self, cache=False, cache_type="ram"):
        from yolox.data import TrainTransform

        return RobustShipDataset(
            data_dir=self.data_dir,
            json_file=self.train_ann,
            name=self.image_folder,
            img_size=self.input_size,
            expected_num_classes=self.num_classes,
            preproc=GrayThermalTransform(
                TrainTransform(max_labels=120, flip_prob=self.flip_prob,
                               hsv_prob=self.hsv_prob),
                gray_prob=TRAIN_GRAY_PROB, invert_prob=TRAIN_INVERT_PROB,
            ),
            cache=cache,
            cache_type=cache_type,
        )

    def get_data_loader(self, batch_size, is_distributed, no_aug=False, cache_img=None):
        # yolox_base.Exp.get_data_loader kopyasi (aerial exp'teki ayni
        # desen): Mosaic 4 goruntuyu birlestirdigi icin KENDI TrainTransform'unu
        # kurar, self.dataset'e verilen preproc'u KULLANMAZ. max_labels bu
        # yuzden burada ayrica belirtilmeli. Tek nesne yogunlugu dusuk (kutu
        # basi ortalama 2,3) ama en yogun goruntu 36 kutu tasiyordu; 4'lu
        # mozaikte teorik tavan ~144 -- aerial'in 4000'i burada gereksiz,
        # 200 yeterli pay birakiyor.
        import torch.distributed as dist

        from yolox.data import (
            TrainTransform,
            YoloBatchSampler,
            DataLoader,
            InfiniteSampler,
            MosaicDetection,
            worker_init_reset_seed,
        )
        from yolox.utils import wait_for_the_master

        if "dataset" not in self.__dict__ or self.dataset is None:
            with wait_for_the_master():
                assert cache_img is None, (
                    "cache_img must be None if you didn't create self.dataset before launch"
                )
                self.dataset = self.get_dataset(cache=False, cache_type=cache_img)

        self.dataset = MosaicDetection(
            dataset=self.dataset,
            mosaic=not no_aug,
            img_size=self.input_size,
            preproc=GrayThermalTransform(
                TrainTransform(max_labels=200, flip_prob=self.flip_prob,
                               hsv_prob=self.hsv_prob),
                gray_prob=TRAIN_GRAY_PROB, invert_prob=TRAIN_INVERT_PROB,
            ),
            degrees=self.degrees,
            translate=self.translate,
            mosaic_scale=self.mosaic_scale,
            mixup_scale=self.mixup_scale,
            shear=self.shear,
            enable_mixup=self.enable_mixup,
            mosaic_prob=self.mosaic_prob,
            mixup_prob=self.mixup_prob,
        )

        if is_distributed:
            batch_size = batch_size // dist.get_world_size()

        sampler = InfiniteSampler(len(self.dataset), seed=self.seed if self.seed else 0)
        batch_sampler = YoloBatchSampler(
            sampler=sampler,
            batch_size=batch_size,
            drop_last=False,
            mosaic=not no_aug,
        )
        dataloader_kwargs = {
            "num_workers": self.data_num_workers,
            "pin_memory": True,
            "batch_sampler": batch_sampler,
            "worker_init_fn": worker_init_reset_seed,
        }
        return DataLoader(self.dataset, **dataloader_kwargs)

    def get_eval_dataset(self, **kwargs):
        from yolox.data import ValTransform

        legacy = kwargs.get("legacy", False)
        dataset = COCODataset(
            data_dir=self.data_dir,
            json_file=self.val_ann,
            name=self.image_folder,
            img_size=self.test_size,
            preproc=(GrayThermalTransform(ValTransform(legacy=legacy), gray_prob=1.0)
                     if self.eval_gray else ValTransform(legacy=legacy)),
        )
        assert_class_scheme(dataset.coco, self.num_classes)
        return dataset

    def get_evaluator(self, batch_size, is_distributed, testdev=False, legacy=False):
        # Aerial projedeki VisDroneEvaluator'dan (ignore-region, top-500 ozel
        # mantigi) BILEREK farkli: bu veri setinde ignore_regions yok, standart
        # COCOEvaluator/COCO mAP yeterli. Karta ozel bir calisma noktasi
        # (deploy_conf'ta F1) gerekirse aerial'daki desen buraya tasinir --
        # simdiden eklemek erken soyutlama olur.
        from yolox.evaluators import COCOEvaluator

        return COCOEvaluator(
            dataloader=self.get_eval_loader(
                batch_size, is_distributed, testdev=testdev, legacy=legacy
            ),
            img_size=self.test_size,
            confthre=self.test_conf,
            nmsthre=self.nmsthre,
            num_classes=self.num_classes,
            testdev=testdev,
        )


In [ ]:
%%writefile ship_metrics.py
#!/usr/bin/env python3
"""Gemi tespitinin son metrikleri: COCO mAP + IoU 0.50'de P/R/F1.

Neden iki ayri motor
--------------------
* **AP** pycocotools ile hesaplanir. Sebep tek: egitim sirasinda YOLOX'un
  COCOEvaluator'u de ayni tanimi kullanir. Boylece Kaggle'daki AP, VM'deki
  float AP ve INT8 AP **ayni sayinin** uc olcumudur; kuantalama kaybi
  ("AP kaybi 0.02'yi gecmesin") ancak boyle anlamli olur. Kendi AP'mizi
  yazsaydik bu uc sayi kiyaslanamazdi.
* **P/R/F1** pycocotools'ta yok. Kart sabit bir guven esiginde calisiyor
  (deploy/src/main.cpp varsayilani 0.15) ve saha sorusu "o esikte kac gemiyi
  kaciriyoruz, kac yanlis alarm veriyoruz" -- bu AP degil, P/R/F1 sorusudur.
  Eslestirme mantigi bu projenin eski `visdrone_eval.py`'sinden devralindi
  (VisDrone'a ozel ignore-region dallari cikarildi; gemi verisinde
  ignore_regions yok, iscrowd hep 0).

AP'nin ve F1'in recall paydasi ayni degildir; ikisi de kendi kuralinca
dogrudur. F1, yayinlanmis YOLO calismalariyla kiyaslanabilsin diye
COCO/Ultralytics kuralini izler: payda gercek GT sayisi.

Kullanim
--------
    from ship_metrics import evaluate_ship, format_metrics
    metrics = evaluate_ship(coco_gt, detections, deploy_conf=0.15)
    print(format_metrics(metrics, "INT8"))
"""

import contextlib
import io
from collections import defaultdict

import numpy as np

#: En iyi F1 aramasinda taranan guven esikleri.
SCORE_GRID = np.round(np.arange(0.0, 1.0001, 0.01), 4)

#: COCO'nun standart maxDets'i. YOLOX COCOEvaluator da bunu kullanir; en
#: kalabalik gemi karesinde 36 kutu olculdu, yani sinir baglayici degil.
MAX_DETS = 100


def _iou_xywh(box_a, box_b):
    ax, ay, aw, ah = (float(v) for v in box_a)
    bx, by, bw, bh = (float(v) for v in box_b)
    iw = min(ax + aw, bx + bw) - max(ax, bx)
    ih = min(ay + ah, by + bh) - max(ay, by)
    if iw <= 0 or ih <= 0:
        return 0.0
    inter = iw * ih
    return inter / max(aw * ah + bw * bh - inter, 1e-12)


def _match_image(gt_boxes, detections, iou_thr=0.50):
    """Bir goruntude tespitleri GT'lere aclikla eslestirir.

    Tespitler skora gore azalan sirada islenir ve her GT en fazla bir kez
    eslesir (COCO ve VisDrone toolkit'inin ortak kurali). Donus: tespit
    basina (skor, 1=TP / 0=FP) ciftleri, skor sirasinda.
    """
    used = [False] * len(gt_boxes)
    rows = []
    for detection in sorted(detections, key=lambda row: -float(row["score"])):
        best_iou = iou_thr
        best_index = None
        for index, gt_box in enumerate(gt_boxes):
            if used[index]:
                continue
            iou = _iou_xywh(detection["bbox"], gt_box)
            if iou >= best_iou:
                best_iou = iou
                best_index = index
        if best_index is None:
            rows.append((float(detection["score"]), 0))
        else:
            used[best_index] = True
            rows.append((float(detection["score"]), 1))
    return rows


def _prf_curves(rows, gt_count):
    """SCORE_GRID uzerinde precision/recall/F1 dizileri.

    `rows` skora gore azalan sirali (skor, eslesme) ciftleridir.
    """
    zeros = np.zeros_like(SCORE_GRID)
    if gt_count <= 0 or not rows:
        return zeros, zeros, zeros

    scores = np.asarray([row[0] for row in rows], dtype=np.float64)
    matches = np.asarray([row[1] for row in rows], dtype=np.int8)
    # scores azalan -> -scores artan; "skor >= t" olan tespit sayisi:
    kept = np.searchsorted(-scores, -SCORE_GRID, side="right")
    tp_cum = np.concatenate(([0.0], np.cumsum(matches == 1, dtype=np.float64)))
    fp_cum = np.concatenate(([0.0], np.cumsum(matches == 0, dtype=np.float64)))
    tp, fp = tp_cum[kept], fp_cum[kept]

    precision = tp / np.maximum(1e-12, tp + fp)
    recall = tp / float(gt_count)
    f1 = 2 * precision * recall / np.maximum(1e-12, precision + recall)
    return precision, recall, f1


def _at(index, precision, recall, f1):
    return {
        "precision": float(precision[index]),
        "recall": float(recall[index]),
        "f1": float(f1[index]),
        "score": float(SCORE_GRID[index]),
    }


def coco_ap(coco_gt, detections, image_ids=None, max_dets=MAX_DETS):
    """(AP@[.50:.95], AP@0.50, AP@0.75) -- pycocotools, egitimle ayni tanim."""
    from pycocotools.cocoeval import COCOeval

    if not detections:
        return 0.0, 0.0, 0.0
    with contextlib.redirect_stdout(io.StringIO()):
        coco_dt = coco_gt.loadRes(list(detections))
        evaluator = COCOeval(coco_gt, coco_dt, "bbox")
        if image_ids is not None:
            evaluator.params.imgIds = list(image_ids)
        evaluator.params.maxDets = [1, 10, max_dets]
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
    return (float(evaluator.stats[0]), float(evaluator.stats[1]),
            float(evaluator.stats[2]))


def prf_at_iou50(coco_gt, detections, deploy_conf, image_ids=None,
                 max_dets=MAX_DETS):
    """Sabit esikte ve en iyi F1 noktasinda P/R/F1.

    Sinif bazinda hesaplanip makro ortalanir; tek sinifli ("ship") veride
    bu dogrudan o sinifin degeridir.
    """
    wanted = set(coco_gt.getImgIds() if image_ids is None else image_ids)
    categories = sorted(getattr(coco_gt, "cats", {}) or {1: None})

    detections_by_key = defaultdict(list)
    for detection in detections:
        image_id = int(detection["image_id"])
        if image_id in wanted:
            detections_by_key[(image_id, int(detection["category_id"]))].append(
                detection)

    gt_by_key = defaultdict(list)
    gt_count = defaultdict(int)
    for image_id in wanted:
        for annotation in coco_gt.imgToAnns.get(image_id, ()):
            if annotation.get("iscrowd", 0):
                continue
            key = (image_id, int(annotation["category_id"]))
            gt_by_key[key].append([float(v) for v in annotation["bbox"]])
            gt_count[int(annotation["category_id"])] += 1

    curves = []
    for category_id in categories:
        rows = []
        for image_id in wanted:
            key = (image_id, category_id)
            image_dets = sorted(detections_by_key.get(key, ()),
                                key=lambda row: -float(row["score"]))[:max_dets]
            rows.extend(_match_image(gt_by_key.get(key, ()), image_dets))
        rows.sort(key=lambda row: -row[0])
        curves.append(_prf_curves(rows, gt_count.get(category_id, 0)))

    if not curves:
        zeros = np.zeros_like(SCORE_GRID)
        curves = [(zeros, zeros, zeros)]
    macro = (np.mean([c[0] for c in curves], axis=0),
             np.mean([c[1] for c in curves], axis=0),
             np.mean([c[2] for c in curves], axis=0))
    fixed_index = int(np.argmin(np.abs(SCORE_GRID - float(deploy_conf))))
    best_index = int(np.argmax(macro[2]))
    return _at(fixed_index, *macro), _at(best_index, *macro)


def evaluate_ship(coco_gt, detections, deploy_conf=0.15, max_dets=MAX_DETS,
                  per_source=True):
    """Tek cagrida teslim edilecek metrikler.

    per_source: goruntulerde `source` alani varsa kaynak bazli AP tablosu da
    uretilir. Tek global sayi bu veri setinde yaniltici olabilir -- kaynaklar
    cok farkli (ir_thermal gercek termal, vais'in %60'i gri ton) ve termal
    AP'nin cokup cokmedigini gosteren tek sey budur.
    """
    ap, ap50, ap75 = coco_ap(coco_gt, detections, max_dets=max_dets)
    at_conf, best = prf_at_iou50(coco_gt, detections, deploy_conf,
                                 max_dets=max_dets)
    metrics = {
        "ap": ap, "ap50": ap50, "ap75": ap75,
        "f1_at": at_conf, "f1_best": best,
        "deploy_conf": float(deploy_conf),
        "n_images": len(coco_gt.getImgIds()),
        "n_detections": len(detections),
        "per_source": {},
    }

    if per_source:
        by_source = defaultdict(list)
        for image_id in coco_gt.getImgIds():
            source = coco_gt.imgs[image_id].get("source")
            if source:
                by_source[str(source)].append(image_id)
        for source, image_ids in sorted(by_source.items()):
            source_ap, source_ap50, _ = coco_ap(coco_gt, detections,
                                                image_ids=image_ids,
                                                max_dets=max_dets)
            metrics["per_source"][source] = {
                "images": len(image_ids), "ap": source_ap, "ap50": source_ap50,
            }
    return metrics


def format_metrics(metrics, title="COCO"):
    """Metrikleri insan okunur tek bir metne cevirir."""
    lines = [
        f"{title}: AP@[.50:.95]={metrics['ap']:.4f}  "
        f"AP@0.50={metrics['ap50']:.4f}  AP@0.75={metrics['ap75']:.4f}"
    ]
    at_conf, best = metrics.get("f1_at"), metrics.get("f1_best")
    if at_conf:
        lines.append(
            f"  Kart esiginde (conf={metrics['deploy_conf']:.2f}): "
            f"P={at_conf['precision']:.4f} R={at_conf['recall']:.4f} "
            f"F1={at_conf['f1']:.4f}"
        )
    if best:
        lines.append(
            f"  En iyi F1={best['f1']:.4f} (P={best['precision']:.4f} "
            f"R={best['recall']:.4f}) esik={best['score']:.2f}"
        )
    per_source = metrics.get("per_source") or {}
    if per_source:
        lines.append(f"  {'kaynak':<20}{'goruntu':>9}{'AP':>9}{'AP50':>9}")
        for source, row in sorted(per_source.items()):
            lines.append(f"  {source:<20}{row['images']:>9}"
                         f"{row['ap']:>9.4f}{row['ap50']:>9.4f}")
    return "\n".join(lines)


In [ ]:
# 6 ham kaynagi bul ve tek sinifli COCO'ya birlestir.
#
# Kaynaklar KATEGORI IMZASINDAN taninir, klasor/dosya adindan DEGIL: Kaggle
# yuklenen zip'i acabilir ya da acmayabilir, ve kullanicinin verdigi dataset
# slug'i onceden bilinmiyor (aerial projedeki 'discover()' ile ayni gerekce
# -- bkz. HANDOFF, "isaret dosyasindan yukari yurume").
import json as _json
import zipfile as _zipfile

INPUT = Path("/kaggle/input")
NEEDED = ("vais_smd_marvel", "singapore_maritime", "sea_vessels",
          "ship_model", "ir_thermal", "wutdet")

# tools/build_ship_dataset.py CLASS_MAPS ile BIREBIR ayni tutulmali (test bunu
# zorunlu kilar). Her kumeyi COCO kategorilerinin TAM kumesiyle karsilastirir
# -- alt-kume degil, tam esitlik: "ship_model"in tek kategorisi "ship" iken
# baska hicbir kaynak kategori kumesini {"ship"}'e daraltmiyor.
CATEGORY_SIGNATURES = {
    "vais_smd_marvel": {"vessel", "buoy", "object"},
    "singapore_maritime": {"objects", "boat", "buoy", "ferry",
                           "flying bird-plane", "kayak", "other",
                           "sail boat", "speed boat", "vessel-ship"},
    "sea_vessels": {"sea-vessels", "fishing boat", "merchant ship",
                    "military ship", "patrol boat", "sails boat",
                    "submarine", "tugboat", "yacht"},
    "ship_model": {"ship"},
    "ir_thermal": {"boat", "bulk carrier", "canoe", "container ship",
                   "fishing boat", "liner", "sailboat", "warship"},
}


def _classify(names):
    lowered = frozenset(str(n).strip().lower() for n in names)
    for key, sig in CATEGORY_SIGNATURES.items():
        if lowered == sig:
            return key
    return None


def _scan_extracted(found):
    # 5 COCO kaynagi: <kok>/<bolum>/_annotations.coco.json
    for js in sorted(INPUT.rglob("_annotations.coco.json")):
        try:
            cats = _json.loads(js.read_text()).get("categories", [])
        except Exception:
            continue
        key = _classify(c.get("name", "") for c in cats)
        if key:
            found.setdefault(key, js.parent.parent)

    # WUTDet: voc/Annotations/*.xml + voc/JPEGImages/ kardes klasorleri.
    # Kok, Annotations'in DOGRUDAN ebeveyni -- tum arsivi degil yalnizca
    # WUTDet'e ait dosyalari kapsasin diye (paylasilan ust klasor hatasi,
    # bkz. aerial projede test_parent_directory_is_not_mistaken_for_a_source).
    for ann in sorted(INPUT.rglob("Annotations")):
        if not ann.is_dir() or not (ann.parent / "JPEGImages").is_dir():
            continue
        if not any(ann.glob("*.xml")):
            continue
        found.setdefault("wutdet", ann.parent)


def _scan_zips(found):
    """Kaggle zip'i acmadiysa arsiv iceriginden tani."""
    for path in sorted(INPUT.rglob("*.zip")):
        try:
            with _zipfile.ZipFile(path) as z:
                names = z.namelist()
                js = [n for n in names
                      if n.rsplit("/", 1)[-1] == "_annotations.coco.json"]
                if js:
                    cats = _json.loads(z.read(sorted(js)[0])).get("categories", [])
                    key = _classify(c.get("name", "") for c in cats)
                    if key:
                        found.setdefault(key, path)
                    continue
                parts = {p for n in names for p in n.split("/")}
                has_xml = any(n.lower().endswith(".xml") for n in names)
                if {"Annotations", "JPEGImages"} <= parts and has_xml:
                    found.setdefault("wutdet", path)
        except Exception:
            continue


def discover_ship_sources():
    found = {}
    _scan_extracted(found)
    if set(NEEDED) - set(found):
        _scan_zips(found)
    missing = [k for k in NEEDED if k not in found]
    if missing:
        listing = "\n".join(f"  {p}" for p in sorted(INPUT.glob("*")))
        raise SystemExit(
            f"Bulunamayan kaynaklar: {missing}\n"
            f"Bagli veri setleri:\n{listing}\n"
            f"Add Input ile eksik olani baglayin."
        )
    return found


SOURCES = discover_ship_sources()
for _name in NEEDED:
    print(f"{_name:<20} {SOURCES[_name]}")

DATASET_DIR = Path(WORK) / "datasets" / "ship_merged"

# Kabuk dizesi yerine argüman listesi: kaynak yollarinda bosluk var
# ("Sea Vessels Dataset...", "ship model..."), kabuk alintilamasi kirilgan olurdu.
_cmd = [sys.executable, "build_ship_dataset.py",
        "--out", str(DATASET_DIR),
        "--images-out", str(DATASET_DIR / "images")]
for _key, _path in SOURCES.items():
    _cmd += ["--source", f"{_key}={_path}"]

_proc = subprocess.run(_cmd, text=True)
assert _proc.returncode == 0, (
    "build_ship_dataset.py basarisiz oldu; yukaridaki cikti hatanin sebebini yazar."
)


In [ ]:
# Baslangic agirligi: YOLOX'un resmi Megvii COCO checkpoint'i (yolox_tiny).
# COCO'da "boat" sinifi var; tek sinifli bu fine-tune icin dogrudan uygun
# bir baslangic noktasi (aerial projedeki ayni gerekce).
import importlib.util
import urllib.request

WDIR = Path(WORK) / "weights"
WDIR.mkdir(exist_ok=True)

spec = importlib.util.spec_from_file_location("exp_mod", EXP_FILE)
exp_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(exp_mod)
ref_state = exp_mod.Exp().get_model().state_dict()

mg = WDIR / INIT_URL.rsplit("/", 1)[-1]
if not mg.exists():
    urllib.request.urlretrieve(INIT_URL, mg)

try:
    raw = torch.load(mg, map_location="cpu", weights_only=False)
except TypeError:
    raw = torch.load(mg, map_location="cpu")
inner = raw.get("model", raw.get("state_dict", raw)) if isinstance(raw, dict) else raw
inner = {(k[7:] if k.startswith("module.") else k): v for k, v in inner.items()}

matched = {
    k: v for k, v in inner.items()
    if k in ref_state and ref_state[k].shape == v.shape
}
missing = sorted(set(ref_state) - set(matched))
allowed_missing = all(
    k == "backbone.backbone.stem.space_to_depth.weight" or ".cls_preds." in k
    for k in missing
)
ratio = len(matched) / max(len(ref_state), 1)
assert ratio >= 0.95 and allowed_missing, (
    f"Baslangic checkpoint'i mimariyle uyumsuz: eslesme={ratio:.1%}, "
    f"beklenmeyen eksikler={missing}"
)

# Tek sinifli head (COCO'nun 80'i degil) ve sabit DPUFocus agirligi yeni
# modelin baslangicindan, diger katmanlar Megvii'den gelir.
init_state = dict(ref_state)
init_state.update(matched)
INIT_CKPT = str(WDIR / "init_ckpt.pth")
torch.save({
    "model": init_state,
    "meta": {"source": str(mg), "yolox_commit": YOLOX_COMMIT,
             "matched_ratio": ratio},
}, INIT_CKPT)
print(f"Megvii baslangici dogrulandi: {ratio:.1%} -> {INIT_CKPT}")
print("Yalnizca cls head (1 sinif) ve sabit DPUFocus katmani yeniden baslatildi.")


## Egitim

In [ ]:
# Tek sinif (ship), girdi 512x512 (kare), 30 epoch -- bkz. yolox_tiny_ship.py
# basligindaki olcum: kaynaklarin %62'si kare, kutu boyutu heterojen.
# 30 epoch TEMKINLI bir baslangic (bu veri icin epoch-AP egrisi henuz
# olculmedi) -- TensorBoard/log'daki AP@0.50 egrisine bakip gerekirse
# ayarlayin.
BATCH = 32  # 512x512 daha kucuk oldugu icin aerial'in 16'sindan yuksek
            # baslanabilir; OOM olursa 16, sonra 8 yapin.

%cd {WORK}
!python YOLOX/tools/train.py -f {EXP_FILE} -d 1 -b {BATCH} --fp16 -c weights/init_ckpt.pth


In [ ]:
# Devam (resume): onceki oturumun Output'unu bu oturuma input olarak
# bagladiktan sonra asagidaki degiskene latest_ckpt.pth yolunu yazin.
RESUME_CKPT = ""  # or. "/kaggle/input/ONCEKI/YOLOX_outputs/yolox_tiny_ship/latest_ckpt.pth"

if RESUME_CKPT:
    !python YOLOX/tools/train.py -f {EXP_FILE} -d 1 -b {BATCH} --fp16 --resume -c "{RESUME_CKPT}"
else:
    print("RESUME_CKPT bos; bu hucre yalnizca yarim kalan egitimi surdurmek icin.")


In [ ]:
# Degerlendirme: standart COCO mAP (bu veri setinde VisDrone'un ignore-region
# kavrami yok, ozel protokole gerek yok -- bkz. yolox_tiny_ship.py get_evaluator).


def find_best_ckpt():
    """Bu oturumda egitildiyse yerelden, aksi halde bagli input'tan alir."""
    local = Path(f"YOLOX_outputs/{Path(EXP_FILE).stem}/best_ckpt.pth")
    if local.exists():
        return local
    if INPUT.exists():
        for candidate in sorted(INPUT.glob("**/best_ckpt.pth")):
            return candidate
    raise SystemExit(
        "best_ckpt.pth bulunamadi. Once egitim hucresini calistirin veya "
        "onceki oturumun ciktisini bu oturuma input olarak baglayin."
    )


BEST_CKPT = str(find_best_ckpt())
print("checkpoint:", BEST_CKPT)
!python YOLOX/tools/eval.py -f {EXP_FILE} -c "{BEST_CKPT}" -d 1 -b 32 --conf 0.001


### Teslim edilecek metrikler

Yukaridaki hucre YOLOX'un standart AP ciktisidir. Asagidaki hucre ayni
checkpoint icin **mAP + kart esiginde P/R/F1 + kaynak bazli tablo** uretir;
ayni `ship_metrics.py` VM'de kuantalama sonrasi da kullanilir, yani float ve
INT8 sayilari ayni tanimla olculur.

Uc set uzerinde birden calisir:

| set | ne icin |
|---|---|
| val | model secimi bu sette yapildi -- iyimser, kiyas icin |
| **test** | **raporlanacak sayi**; egitim/ayar bu seti hic gormedi |
| val (gri) | gri-ton dayanikliligi: AP dususu dayanikliligin olcusu |


In [ ]:
# mAP + F1 + kaynak bazli AP.
# Kaynak bazli tablo neden: kaynaklar cok farkli (ir_thermal GERCEK termal,
# vais'in %60'i gri ton, digerleri RGB). Tek global sayi, termalin cokmesini
# gizleyebilir -- termal/gri dayaniklilik sarti ancak o satirdan dogrulanir.
import contextlib
import importlib.util
import io as _io

import cv2
import numpy as np
import torch
from pycocotools.coco import COCO

from yolox.data import ValTransform
from yolox.utils import postprocess

from ship_metrics import evaluate_ship, format_metrics

_BGR_LUMA = np.array([0.114, 0.587, 0.299], dtype=np.float32).reshape(3, 1, 1)

spec3 = importlib.util.spec_from_file_location("exp_mod3", EXP_FILE)
_mod3 = importlib.util.module_from_spec(spec3)
spec3.loader.exec_module(_mod3)
_exp = _mod3.Exp()
_model = _exp.get_model().eval()
try:
    _ckpt = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)
except TypeError:
    _ckpt = torch.load(BEST_CKPT, map_location="cpu")
_model.load_state_dict(_ckpt["model"], strict=True)
_device = "cuda" if torch.cuda.is_available() else "cpu"
_model.to(_device)
_transform = ValTransform(legacy=False)


def detect_all(ann_name, gray=False, batch=16, conf=0.001):
    """Egitimdeki ValTransform'un aynisi; gray=True ise BT.601 parlakligi
    (GrayThermalTransform ile ayni formul)."""
    with contextlib.redirect_stdout(_io.StringIO()):
        coco = COCO(str(DATASET_DIR / "annotations" / ann_name))
    images = coco.dataset["images"]
    results = []
    for start in range(0, len(images), batch):
        chunk = images[start:start + batch]
        tensors, ratios = [], []
        for meta in chunk:
            raw = cv2.imread(str(DATASET_DIR / "images" / meta["file_name"]))
            img, _ = _transform(raw, None, _exp.test_size)
            if gray:
                img = np.repeat((img * _BGR_LUMA).sum(axis=0, keepdims=True), 3, axis=0)
            tensors.append(torch.from_numpy(np.ascontiguousarray(img)))
            ratios.append(min(_exp.test_size[0] / raw.shape[0],
                              _exp.test_size[1] / raw.shape[1]))
        with torch.no_grad():
            out = _model(torch.stack(tensors).float().to(_device))
            out = postprocess(out, _exp.num_classes, conf, _exp.nmsthre)
        for meta, ratio, dets in zip(chunk, ratios, out):
            if dets is None:
                continue
            for *box, obj_conf, cls_conf, _cls in dets.cpu().numpy():
                x1, y1, x2, y2 = [v / ratio for v in box]
                results.append({"image_id": meta["id"], "category_id": 1,
                                "bbox": [float(x1), float(y1),
                                         float(x2 - x1), float(y2 - y1)],
                                "score": float(obj_conf * cls_conf)})
        if (start // batch) % 20 == 0:
            print(f"  {min(start + batch, len(images))}/{len(images)}", end="\r")
    return coco, results


def report(ann_name, gray=False):
    coco, dets = detect_all(ann_name, gray=gray)
    if not dets:
        print(f"{ann_name}: hic tespit yok -- checkpoint veya esik yanlis olabilir.")
        return None
    metrics = evaluate_ship(coco, dets, deploy_conf=_exp.deploy_conf)
    label = f"{ann_name}{' [GRI]' if gray else ''}"
    print(format_metrics(metrics, label))
    print()
    return metrics["ap"]


ap_val = report("instances_val.json")
ap_test = report("instances_test.json")          # RAPORLANACAK SAYI
ap_gray = report("instances_val.json", gray=True)

print("=" * 66)
if ap_val and ap_test:
    print(f"val {ap_val:.4f} -> test {ap_test:.4f}  "
          f"(fark {ap_test - ap_val:+.4f}; val model seciminden dolayi iyimser)")
if ap_val and ap_gray:
    print(f"GRI DAYANIKLILIGI: {ap_val:.4f} -> {ap_gray:.4f} "
          f"({(ap_gray - ap_val) / ap_val * 100:+.1f}%)")
print("VM'de kuantalama sonrasi ayni modulle olcun; sayilar kiyaslanabilir.")


In [ ]:
# Gorsel kontrol: kutular + MERKEZ NOKTALARI (cx, cy)
# KV260 uygulamasindaki hesabin aynisi: cx=(x1+x2)/2, cy=(y1+y2)/2
import json
import random

import cv2
import matplotlib.pyplot as plt

from yolox.data.data_augment import ValTransform
from yolox.utils import postprocess

spec2 = importlib.util.spec_from_file_location("exp_mod2", EXP_FILE)
exp_mod2 = importlib.util.module_from_spec(spec2)
spec2.loader.exec_module(exp_mod2)
exp = exp_mod2.Exp()
CLASSES = exp_mod2.TARGET_CLASSES

model = exp.get_model().eval()
try:
    ckpt = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)
except TypeError:
    ckpt = torch.load(BEST_CKPT, map_location="cpu")
model.load_state_dict(ckpt["model"], strict=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

CONF_VIS = 0.15  # kartla ayni dagitim esigi
val_transform = ValTransform(legacy=False)

# Her kaynaktan birer ornek: 6 kaynagin da nasil goruldugunu izleyelim
val_json = json.loads(
    (DATASET_DIR / "annotations" / "instances_val.json").read_text())
by_source = {}
for _image in val_json["images"]:
    by_source.setdefault(_image["source"], []).append(_image["file_name"])
random.seed(0)
sample_names = [random.choice(v) for v in by_source.values()]

fig, axes = plt.subplots(len(sample_names), 1,
                         figsize=(10, 8 * len(sample_names)))
axes = np.atleast_1d(axes)
for ax, name in zip(axes, sample_names):
    img0 = cv2.imread(str(DATASET_DIR / "images" / name))
    h0, w0 = img0.shape[:2]
    ratio = min(exp.test_size[0] / h0, exp.test_size[1] / w0)
    img, _ = val_transform(img0, None, exp.test_size)
    with torch.no_grad():
        out = model(torch.from_numpy(img).unsqueeze(0).float().to(device))
        out = postprocess(out, exp.num_classes, CONF_VIS, exp.nmsthre)[0]
    vis = img0.copy()
    if out is not None:
        for *box, obj_conf, cls_conf, cls_id in out.cpu().numpy():
            x1, y1, x2, y2 = [v / ratio for v in box]
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            score = obj_conf * cls_conf
            cv2.rectangle(vis, (int(x1), int(y1)), (int(x2), int(y2)),
                          (0, 255, 0), 2)
            cv2.circle(vis, (int(cx), int(cy)), 4, (0, 0, 255), -1)
            cv2.putText(
                vis,
                f"{CLASSES[int(cls_id)]} {score:.2f} ({int(cx)},{int(cy)})",
                (int(x1), max(14, int(y1) - 5)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Oracle VM'e tasinacak dosyalari paketle
import shutil

ART = Path(WORK) / "artifacts"
ART.mkdir(exist_ok=True)
shutil.copy(BEST_CKPT, ART / "best_ckpt.pth")
for _name in ("yolox_tiny_ship.py", "build_ship_dataset.py",
              "dataset_common.py", "ship_metrics.py"):
    shutil.copy(_name, ART / _name)
# Kuantalama train disindan kalibre edilmemeli: train anotasyonu PTQ
# kalibrasyonunu sinirlar, val anotasyonu ise yalniz accuracy gate icindir.
shutil.copy(DATASET_DIR / "annotations" / "instances_train.json",
            ART / "instances_train.json")
shutil.copy(DATASET_DIR / "annotations" / "instances_val.json",
            ART / "instances_val.json")
# Nihai sayi test setinde raporlanir (quantize_yolox.py --report-only)
shutil.copy(DATASET_DIR / "annotations" / "instances_test.json",
            ART / "instances_test.json")
(ART / "classes.txt").write_text("\n".join(CLASSES))
(ART / "YOLOX_COMMIT.txt").write_text(YOLOX_COMMIT + "\n")
(ART / "EXP_FILE.txt").write_text(EXP_FILE + "\n")
zip_path = shutil.make_archive(str(Path(WORK) / "yolox_ship_artifacts"),
                               "zip", ART)
print("Hazir:", zip_path)
print("Not defteri kaydedilince Output sekmesinden indirebilirsiniz.")
print("VM'e ayrica val goruntuleri lazim:", DATASET_DIR / "images")


## Sonraki adim: Oracle VM'de kuantalama

`yolox_ship_artifacts.zip` dosyasini ve `datasets/ship_merged/images` klasorunu
VM'e tasiyin, ardindan `quantize/README.md` adimlarini izleyin (exp dosyasi
olarak `yolox_tiny_ship.py`, sinif sayisi 1).
